# 03 - Cohort allele frequencies and variation absent from reference panels

![cohort](https://img.shields.io/badge/cohort-KZ--219-1b9e77)
![platform](https://img.shields.io/badge/platform-DNBSEQ--G400-7570b3)
![pipeline](https://img.shields.io/badge/DRAGEN-v3.10.16-d95f02)
![reference](https://img.shields.io/badge/reference-GRCh37%2Fhg19-e7298a)
![annotation](https://img.shields.io/badge/ANNOVAR-v2025Mar02-666666)
![bioproject](https://img.shields.io/badge/BioProject-PRJNA1224597-333333)

| | |
|---|---|
| **Cohort** | KZ-219 - 219 healthy Kazakh adults |
| **Sequencing** | DNBSEQ-G400, PCR-free, 100 bp paired-end |
| **Alignment and variant calling** | Illumina DRAGEN Bio-IT Platform v3.10.16 |
| **Reference** | GRCh37 / hg19 (DRAGEN hash table v8) |
| **Annotation** | ANNOVAR v2025Mar02 |
| **Raw data** | NCBI SRA, BioProject PRJNA1224597 |

| | |
|---|---|
| Sex / ploidy |  expected ploidy per region (autosome, PAR, X non-PAR, Y non-PAR) |
| chrY | excluded from allele-frequency and absence analysis |
| Headline count | SNV, AC ≥ 2, call rate ≥ 90 %, resolved allele index |
| Unresolved multiallelic rows | flagged and excluded, counted in Methods |
| Terminology | absent from the queried reference panels |

### Running

Inputs, set by environment variables (defaults relative to the repository root):

| Variable | Default | Content |
|---|---|---|
| `KZ_CALLSET` | `data/kz_219_hg19_dragen.annotated.header.txt` | ANNOVAR-annotated joint callset (tab-separated, VCF `REF`, `ALT`, `FORMAT` and one column per sample) |
| `KZ_SEX_FILE` | `data/kz_219_tribes.tsv` | sample metadata with `IID` and `Gender` |
| `KZ_WORK_DIR` | `work/03_af` | large intermediate tables and the scan cache |
| `KZ_OUT_DIR` | `results/03_af` | figure, Supplementary Data 3, Supplementary Tables |

`kz219_style.py` (shared plotting helpers) sits next to the notebook.

```bash
export KZ_CALLSET=data/kz_219_hg19_dragen.annotated.header.txt
export KZ_SEX_FILE=data/kz_219_tribes.tsv
pip install -r requirements.txt
jupyter nbconvert --to notebook --execute --inplace 03_allele_frequencies.ipynb
```

The single pass over a full 30-million-row callset takes hours and writes tables of several
GB to `KZ_WORK_DIR`; the scan state is cached there, so later runs rebuild the tables and the
figure in seconds. Set `DRY_RUN_ROWS` for a quick test on the first rows.

Individual genotypes and the sample metadata are not part of the repository; raw reads are in
NCBI SRA under BioProject PRJNA1224597.

## 1. Configuration

**Absence rules.** Four rules are evaluated in the same pass so that the disagreement
between them is a measured quantity rather than a choice made out of sight.

| Rule | Databases | Role |
|---|---|---|
| **A** | gnomAD genomes and exomes, 1000 Genomes, GME Variome | reported |
| B | A plus ExAC, HRC, Kaviar, PopFreqMax | sensitivity |
| C | B plus dbSNP, COSMIC, ClinVar presence | defines the file split read by notebook 04 |
| D | gnomAD genomes only | sensitivity |

Absence is defined by database presence, not by frequency: a reported frequency of
exactly zero still means the site was typed.

**Restriction cascade.** The reported set is rule A **and** all of: single-nucleotide
substitution, `AC_pass >= 2`, `call_rate >= 0.90` against the expected ploidy,
allele index resolved, chromosome in autosomes or chrX. Each step is counted separately
so the cascade can be shown in full. Indels are excluded from the headline because
representation between ANNOVAR normalisation and the reference panels does not match,
which inflates their apparent absence; they are reported separately.

**Sex.** Expected ploidy drives both the call-rate denominator and the expected allele
number. Metadata sex is self-reported, so the DRAGEN
ploidy call is authoritative and the two are cross-tabulated in `SexConcordance`.

In [ ]:
import gc
import json
import os
import pickle
import platform
import shutil
import sys
from datetime import datetime

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from IPython.display import display
from scipy import stats
from tqdm import tqdm

from kz219_style import (apply_style, COL, CYCLE, FIG_W_MM, MM, save_fig,
                         label_panels, check_min_font, lollipop, wilson_ci,
                         log2fc_ci, hwe_exact_p, bh)

FONT = apply_style()

from matplotlib.text import Text
from PIL import Image
import kz219_style as _ks

FS, FS_PANEL, FS_MAX = 8, 10, 12
DPI, FIG_FORMATS = 1200, ('tiff', 'png')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'Liberation Sans', 'DejaVu Sans'],
    'font.size': FS, 'axes.labelsize': FS, 'axes.titlesize': FS,
    'xtick.labelsize': FS, 'ytick.labelsize': FS,
    'legend.fontsize': FS, 'legend.title_fontsize': FS,
    'hatch.linewidth': 0.5, 'pdf.fonttype': 42, 'ps.fonttype': 42,
    'figure.facecolor': 'white', 'savefig.facecolor': 'white',
})
FONT = plt.rcParams['font.sans-serif'][0]

SUPP_NUM = {
    'fig_absence_sensitivity': 6,
    'tab_composition': 7,     
    'tab_parsing': 8, 'tab_rules': 9, 'tab_titv': 10, 'tab_spectrum': 11,
    'tab_artefact': 12, 'tab_functional': 13, 'tab_indel': 14,
    'tab_refdisc': 15, 'tab_correlation': 16, 'tab_maf': 17,
    'data': 3,
}

_ks_label_panels = _ks.label_panels


def label_panels(fig, axs, **kw):
    """kz219_style.label_panels, with the letters forced to 10 pt bold."""
    axes = list(np.ravel(axs))
    before = {id(t) for t in list(fig.texts) + [t for a in axes for t in a.texts]}
    out = _ks_label_panels(fig, axs, **kw)
    for t in list(fig.texts) + [t for a in axes for t in a.texts]:
        if id(t) not in before:
            t.set_fontsize(FS_PANEL); t.set_fontweight('bold')
    return out


def _enforce_text(fig):
    changed = []
    for t in fig.findobj(Text):
        if not t.get_visible() or not t.get_text().strip():
            continue
        s = t.get_fontsize()
        if s < FS - 1e-6 or s > FS_MAX + 1e-6:
            changed.append((t.get_text()[:30], s))
            t.set_fontsize(min(max(s, FS), FS_MAX))
    return changed


def save_fig(fig, stem, outdir='.', dpi=DPI, formats=FIG_FORMATS):
    """PNG (RGB) and TIFF (CMYK, LZW) at 1200 dpi; text below 8 pt is raised and reported."""
    changed = _enforce_text(fig)
    if changed:
        print(f'  NOTE {stem}: {len(changed)} text item(s) raised to {FS} pt, check the layout: '
              + ', '.join(f"'{t}' ({s:g})" for t, s in changed[:6]))
    png = os.path.join(outdir, f'{stem}.png')
    fig.savefig(png, dpi=dpi, bbox_inches='tight', pad_inches=0.03)
    paths = []
    if 'tiff' in formats:
        tif = os.path.join(outdir, f'{stem}.tiff')
        with Image.open(png) as im:
            im.convert('RGB').convert('CMYK').save(tif, compression='tiff_lzw', dpi=(dpi, dpi))
        paths.append(tif)
    if 'png' in formats:
        paths.append(png)
    else:
        os.remove(png)
    for p in paths:
        with Image.open(p) as im:
            w, h, mode = *im.size, im.mode
        print(f'  saved {os.path.basename(p)}: {mode}, {w / dpi * 25.4:.0f} x {h / dpi * 25.4:.0f} mm '
              f'at {dpi} dpi, {os.path.getsize(p) / 1e6:.1f} MB')
    return paths

VCF = os.environ.get('KZ_CALLSET', 'data/kz_219_hg19_dragen.annotated.header.txt')
SEX_FILE = os.environ.get('KZ_SEX_FILE', 'data/kz_219_tribes.tsv')
PLOIDY_FILE = None

WORK_DIR = os.environ.get('KZ_WORK_DIR', 'work/03_af')      # large intermediate tables
OUT_DIR = os.environ.get('KZ_OUT_DIR', 'results/03_af')
STEM = 'kz_219_hg19_AF_from_GT'

AF_FILE = os.path.join(WORK_DIR, f'{STEM}.txt')
NOVEL_FILE = os.path.join(WORK_DIR, f'{STEM}_absent_ruleC.txt')
KNOWN_FILE = os.path.join(WORK_DIR, f'{STEM}_known.txt') 
CMP_FILE = os.path.join(WORK_DIR, 'kz_219_AF_comparison.txt')
DISC_FILE = os.path.join(WORK_DIR, 'kz_219_hg19_refdiscordant_80pct.txt')
CACHE_FILE = os.path.join(WORK_DIR, f'{STEM}_scan_state.pkl')
CACHE_DONE = CACHE_FILE + '.done.json'
XLSX = os.path.join(OUT_DIR, 'KZ219_allele_frequencies.xlsx')
GENES_EXONIC = os.path.join(OUT_DIR, 'refdiscordant_exonic_genes.txt')

for _p in (VCF, SEX_FILE):
    assert os.path.exists(_p), f'input not found: {_p} (see the Running section)'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

CHUNKSIZE = 100_000
DRY_RUN_ROWS = None
OVERWRITE = True
USE_CACHE = True        
WRITE_FULL_AF_TABLE = False 

COHORT = 'KZ-219'
N_COHORT = 219
CALLER = 'DRAGEN gVCF v3.10.16, joint genotyping v4.3.6'
ANNOVAR_BUILD = 'ANNOVAR release 2025Mar02, refGene'

MIN_CALL_RATE = 0.90
MIN_AC_REPORTED = 2
ENRICHMENT_REF = 'gnomAD_popmax'
ENRICHMENT_FOLD = 5.0
ENRICH_MIN_AC = 5
ENRICH_LO, ENRICH_HI = 0.01, 0.99
DISCORDANT_THRESHOLD = 0.80
RARE_MAF = 0.01
HWE_ALPHA = 1e-6
HWE_MIN_MAF = 0.01     

FREQ_PANELS = {
    'gnomAD_ALL': 'AF.1', 'gnomAD_popmax': 'AF_popmax.1', 'gnomAD_EAS': 'AF_eas.1',
    'gnomAD_NFE': 'AF_nfe.1', 'gnomAD_AFR': 'AF_afr.1', 'gnomAD_FIN': 'AF_fin.1',
    'gnomAD_AMR': 'AF_amr.1', 'gnomAD_SAS': 'AF_sas.1',
    '1KG_ALL': '1000G_ALL', '1KG_EAS': '1000G_EAS', '1KG_EUR': '1000G_EUR',
    '1KG_SAS': '1000G_SAS', '1KG_AFR': '1000G_AFR', '1KG_AMR': '1000G_AMR',
    'GME_ALL': 'GME_AF',
}

NOVELTY_CORE = [
    'AF.1', 'AF_popmax.1', 'AF_eas.1', 'AF_nfe.1', 'AF_afr.1', 'AF_fin.1',
    'AF_amr.1', 'AF_sas.1',
    'AF', 'AF_popmax', 'AF_eas', 'AF_nfe', 'AF_afr', 'AF_amr', 'AF_sas',
    '1000G_ALL', '1000G_AFR', '1000G_AMR', '1000G_EAS', '1000G_EUR', '1000G_SAS',
    'GME_AF',
]
NOVELTY_EXTRA = ['ExAC_ALL', 'ExAC_nonpsych_ALL', 'HRC_AF', 'Kaviar_AF', 'PopFreqMax']
CATALOGUE_COLS = ['avsnp151', 'cosmic70', 'CLNSIG']
DESCRIPTIVE_COLS = ['CLNDN', 'CLNREVSTAT']

RULES = {
    'A_core_frequency': NOVELTY_CORE,
    'B_all_frequency': NOVELTY_CORE + NOVELTY_EXTRA,
    'C_with_catalogue': NOVELTY_CORE + NOVELTY_EXTRA + CATALOGUE_COLS,
    'D_gnomAD_genomes': ['AF.1', 'AF_popmax.1', 'AF_eas.1', 'AF_nfe.1',
                         'AF_afr.1', 'AF_fin.1', 'AF_amr.1', 'AF_sas.1'],
}
RULE_TEXT = {
    'A_core_frequency': 'gnomAD genomes and exomes, 1000 Genomes, GME Variome',
    'B_all_frequency': 'rule A plus ExAC, HRC, Kaviar, PopFreqMax',
    'C_with_catalogue': 'rule B plus dbSNP, COSMIC and ClinVar presence',
    'D_gnomAD_genomes': 'gnomAD genomes only',
}
REPORTED_RULE = 'A_core_frequency'
FILE_SPLIT_RULE = 'C_with_catalogue'

MISSING_TOKENS = {'.', '', 'nan', 'NaN', 'NA', 'None', '-'}

CORR_PANELS = ['gnomAD_EAS', 'gnomAD_NFE', 'gnomAD_SAS', 'gnomAD_ALL',
               '1KG_EAS', '1KG_EUR', '1KG_SAS', 'GME_ALL']
SPECTRUM_PANELS = ['gnomAD_EAS', 'gnomAD_NFE', '1KG_SAS']

MAF_EDGES = np.array([0.001, 0.01, 0.05, 0.5])
MAF_LABELS = ['0.1 to 1%', '1 to 5%', '>=5%']
KZ_MAF_EDGES = np.array([0.0, 0.001, 0.01, 0.05, 0.5])
KZ_MAF_LABELS = ['<0.1%', '0.1 to 1%', '1 to 5%', '>=5%']
DISC_MAF_EDGES = np.array([0.0, 0.001, 0.01, 0.05, 0.2, 0.5])
DISC_MAF_LABELS = ['<0.1%', '0.1 to 1%', '1 to 5%', '5 to 20%', '20 to 50%']

CORR_MAX_N = 2_000_000
RNG = np.random.default_rng(0)

# genome
PAR_HG19 = {
    'X': [(60001, 2699520), (154931044, 155260560)],
    'Y': [(10001, 2649520), (59034050, 59363566)],
}
HG19_LEN = {
    '1': 249250621, '2': 243199373, '3': 198022430, '4': 191154276, '5': 180915260,
    '6': 171115067, '7': 159138663, '8': 146364022, '9': 141213431, '10': 135534747,
    '11': 135006516, '12': 133851895, '13': 115169878, '14': 107349540,
    '15': 102531392, '16': 90354753, '17': 81195210, '18': 78077248, '19': 59128983,
    '20': 63025520, '21': 48129895, '22': 51304566, 'X': 155270560, 'Y': 59373566,
    'MT': 16569,
}
AUTOSOMES = [str(i) for i in range(1, 23)]
CHROM_ORDER = AUTOSOMES + ['X', 'Y', 'MT']
ANALYSIS_CHROMS = set(AUTOSOMES) | {'X'}

DERIVED_COLS = ['AF_all', 'AF_pass', 'MAF_all', 'MAF_pass',
                'AC_all', 'AN_all', 'AC_pass', 'AN_pass',
                'n_het_pass', 'n_hom_alt_pass', 'n_hemi_alt_pass',
                'hom_alt_freq_pass', 'call_rate_pass', 'AN_expected',
                'region_class', 'alt_index', 'alt_index_unresolved', 'HWE_p']
ANNOT_COLS = (['Chr', 'Start', 'End', 'Ref', 'Alt', 'Func.refGene', 'Gene.refGene',
               'ExonicFunc.refGene', 'AAChange.refGene']
              + CATALOGUE_COLS + DESCRIPTIVE_COLS
              + sorted(set(NOVELTY_CORE + NOVELTY_EXTRA)))
META_COLS = ANNOT_COLS[:9] + DERIVED_COLS + ANNOT_COLS[9:]
DISC_COLS = ['Chr', 'Start', 'End', 'Ref', 'Alt', 'Func.refGene', 'Gene.refGene',
             'ExonicFunc.refGene', 'AAChange.refGene', 'avsnp151',
             'AF_pass', 'MAF_pass', 'AC_pass', 'AN_pass', 'call_rate_pass',
             'n_hom_alt_pass', 'hom_alt_freq_pass', 'CLNSIG',
             'gnomAD_ALL', 'gnomAD_EAS', 'gnomAD_NFE', '1KG_EAS', '1KG_EUR']

XL_ROW_LIMIT = 200_000
MANIFEST = []


def register(stage, role, path, note=''):
    exists = os.path.exists(path)
    MANIFEST.append({'Stage': stage, 'Role': role, 'File': os.path.basename(path),
                     'Exists': exists,
                     'Size (GB)': round(os.path.getsize(path) / 1024 ** 3, 3) if exists else 0,
                     'Note': note})
    return path


print(f'cohort   : {COHORT}, N = {N_COHORT}')
print(f'reported : rule {REPORTED_RULE}, restricted to SNV, AC >= {MIN_AC_REPORTED}, '
      f'call rate >= {MIN_CALL_RATE:.0%}, autosomes and chrX')
print(f'font     : {FONT}')

## 2. Sex and expected ploidy

`Gender` in the metadata is self-reported. The DRAGEN ploidy estimate is authoritative; the
metadata is kept only as a concordance check, which is what `SexConcordance` reports.

Expected allele number per region:

| Region | Individuals | Expected AN |
|---|---|---|
| Autosome, PAR1, PAR2 | 219 | 438 |
| chrX outside PAR | 219 | 2 × 145 + 74 = 364 |
| chrY outside PAR | 74 | 74 |

In [ ]:
EXPECTED_SEX_COUNTS = {'XX': 145, 'XY': 74}


def load_sex(meta_path, ploidy_path=None, override=None):
    meta = pd.read_csv(meta_path, sep='\t', dtype=str)
    assert {'IID', 'Gender'} <= set(meta.columns), 'IID and Gender required'
    reported = (meta['Gender'].astype('string').str.strip().str.lower()
                .map({'female': 'XX', 'male': 'XY'}))
    sex = pd.Series(reported.to_numpy(), index=meta['IID'].to_numpy(), dtype=object)
    source = pd.Series(np.where(reported.notna(), 'self-reported', 'not recorded'),
                       index=meta['IID'].to_numpy(), dtype=object)
    called = pd.Series(pd.NA, index=sex.index, dtype=object)

    if ploidy_path and os.path.exists(ploidy_path):
        pl = pd.read_csv(ploidy_path, sep='\t', dtype=str)
        col = next(c for c in pl.columns if c.lower() in ('ploidy_sex', 'sex', 'ploidy'))
        pl = pl.set_index('IID')[col].str.upper().str.strip()
        called.loc[pl.index.intersection(called.index)] = pl
        hit = called.notna()
        sex[hit] = called[hit]
        source[hit] = 'DRAGEN ploidy'

    for iid, val in (override or {}).items():
        assert iid in sex.index, f'{iid} not in {os.path.basename(meta_path)}'
        called[iid] = val
        sex[iid] = val
        source[iid] = 'DRAGEN ploidy'

    concord = pd.DataFrame({'Reported': reported.to_numpy(), 'Called': called.to_numpy(),
                            'Used': sex.to_numpy(), 'Source': source.to_numpy()},
                           index=sex.index)
    return sex, concord


SEX, sex_concord = load_sex(register('sex', 'input', SEX_FILE), PLOIDY_FILE)
N_XX = int((SEX == 'XX').sum())
N_XY = int((SEX == 'XY').sum())

assert len(SEX) == N_COHORT, f'{len(SEX)} sex records, expected {N_COHORT}'
assert SEX.notna().all(), f'sex unresolved for: {list(SEX[SEX.isna()].index)}'
assert (N_XX, N_XY) == (EXPECTED_SEX_COUNTS['XX'], EXPECTED_SEX_COUNTS['XY']), \
    f'{N_XX} XX / {N_XY} XY, expected {EXPECTED_SEX_COUNTS}'

AN_AUTO = 2 * N_COHORT
AN_X_NONPAR = 2 * N_XX + N_XY
AN_Y_NONPAR = N_XY

REGION_N_IND = {'auto': N_COHORT, 'X_par': N_COHORT, 'X_nonpar': N_COHORT,
                'Y_par': N_COHORT, 'Y_nonpar': N_XY, 'MT': N_COHORT, 'other': N_COHORT}
REGION_AN = {'auto': AN_AUTO, 'X_par': AN_AUTO, 'X_nonpar': AN_X_NONPAR,
             'Y_par': AN_AUTO, 'Y_nonpar': AN_Y_NONPAR, 'MT': N_COHORT,
             'other': AN_AUTO}
REGION_CODES = ['auto', 'X_par', 'X_nonpar', 'Y_par', 'Y_nonpar', 'MT', 'other']
_CODE = {name: i for i, name in enumerate(REGION_CODES)}
_N_IND_VEC = np.array([REGION_N_IND[c] for c in REGION_CODES], dtype=np.int32)
_AN_VEC = np.array([REGION_AN[c] for c in REGION_CODES], dtype=np.int32)


def region_code(chrom, pos):
    """Vectorised region class for a chunk: autosome, PAR, X/Y outside PAR, MT."""
    chrom = np.asarray(chrom, dtype=object)
    pos = np.asarray(pos, dtype=np.int64)
    out = np.full(chrom.shape, _CODE['other'], dtype=np.int8)
    out[np.isin(chrom, AUTOSOMES)] = _CODE['auto']
    out[chrom == 'MT'] = _CODE['MT']
    for c in ('X', 'Y'):
        m = chrom == c
        if not m.any():
            continue
        in_par = np.zeros(m.sum(), dtype=bool)
        for lo, hi in PAR_HG19[c]:
            in_par |= (pos[m] >= lo) & (pos[m] <= hi)
        sub = np.where(in_par, _CODE[f'{c}_par'], _CODE[f'{c}_nonpar']).astype(np.int8)
        out[m] = sub
    return out


sex_summary = pd.DataFrame([
    ('Individuals', N_COHORT),
    ('XX', N_XX), ('XY', N_XY),
    ('Sex from DRAGEN ploidy', int((sex_concord['Source'] == 'DRAGEN ploidy').sum())),
    ('Sex self-reported only', int((sex_concord['Source'] == 'self-reported').sum())),
    ('Metadata sex missing', int(sex_concord['Reported'].isna().sum())),
    ('Reported and called disagree',
     int((sex_concord['Reported'].notna() & sex_concord['Called'].notna()
          & (sex_concord['Reported'] != sex_concord['Called'])).sum())),
    ('Expected AN, autosome and PAR', AN_AUTO),
    ('Expected AN, chrX outside PAR', AN_X_NONPAR),
    ('Expected AN, chrY outside PAR', AN_Y_NONPAR),
], columns=['Item', 'Value'])
display(sex_summary)

In [ ]:
STAGE = 'preflight'
assert os.path.exists(VCF), f'input not found: {VCF}'
register(STAGE, 'input', VCF, 'annotated callset')

header = list(pd.read_csv(VCF, sep='\t', nrows=0).columns)
assert 'FORMAT' in header, 'FORMAT column not found, cannot locate the sample block'
sample_cols = header[header.index('FORMAT') + 1:]

checks = []


def check(name, ok, detail=''):
    checks.append({'Check': name, 'Result': 'pass' if ok else 'FAIL', 'Detail': detail})
    return ok


check('sample columns', len(sample_cols) == N_COHORT,
      f'{len(sample_cols)} found, {N_COHORT} expected')
missing_sex = sorted(set(sample_cols) - set(SEX.index))
extra_sex = sorted(set(SEX.index) - set(sample_cols))
check('sample names match the sex table', not missing_sex and not extra_sex,
      f'{len(missing_sex)} without sex, {len(extra_sex)} unused'
      + (f'; first missing: {missing_sex[:3]}' if missing_sex else ''))

missing_panels = [f'{k} -> {v}' for k, v in FREQ_PANELS.items() if v not in header]
check('frequency panel columns', not missing_panels, ', '.join(missing_panels) or 'all present')
check('enrichment reference', FREQ_PANELS.get(ENRICHMENT_REF) in header, ENRICHMENT_REF)
for rule, cols in RULES.items():
    present = [c for c in cols if c in header]
    check(f'rule {rule}', len(present) > 0,
          f'{len(present)} of {len(cols)} columns present')
missing_meta = [c for c in ANNOT_COLS if c not in header]
check('annotation columns', True, f'absent and skipped: {missing_meta or "none"}')
check('VCF REF/ALT present for allele resolution',
      'REF' in header and 'ALT' in header,
      'without them multiallelic rows cannot be resolved')

src_gb = os.path.getsize(VCF) / 1024 ** 3
free_gb = shutil.disk_usage(WORK_DIR).free / 1024 ** 3
need = src_gb * ((1 if WRITE_FULL_AF_TABLE else 0) + 1 + 0.05)
check('free disk space', free_gb > need, f'{free_gb:.0f} GB free, {need:.0f} GB needed')

preflight = pd.DataFrame(checks)
display(preflight)
failed = preflight[preflight['Result'] == 'FAIL']
assert failed.empty, f'preflight failed:\n{failed.to_string(index=False)}'
print(f'preflight clean; source callset {src_gb:.1f} GB')

## 4. Genotype parsing and allele-index resolution

Genotypes are parsed column-wise with fixed-width NumPy string operations. `FT` is read
only when `FORMAT` declares it; when absent the genotype is treated as passing. A
haploid call contributes one chromosome to `AN`, so a hemizygous chrX or chrY call is
never counted as a homozygote.

For each ANNOVAR row the alternate allele is matched to the VCF `ALT` list by normalised
allele pair. ANNOVAR emits one row per alternate allele while genotypes keep the original
VCF indices, so counting any non-reference allele assigns both alternates to both rows and
double counts every `1/2` genotype. Rows where the index cannot be resolved are flagged
and excluded rather than defaulted to index 1.

In [ ]:
GT_WIDTH = 8
CELL_WIDTH = 128
FT_OK = ('PASS', '.', '')


def _codes(a):
    out = np.full(a.shape, -2, dtype=np.int16)
    m = np.char.isdigit(a)
    if m.any():
        out[m] = a[m].astype(np.int16)
    return out


def parse_gt_block(sample_block, format_field):
    """(alleles, ploidy, ft_pass) for one chunk; -2 marks a no-call."""
    raw = sample_block.to_numpy(dtype=object)
    n_rows, n_samples = raw.shape
    fmt = np.asarray([str(f) for f in format_field])
    uniq, inv = np.unique(fmt, return_inverse=True)
    ft_index = {i: (u.split(':').index('FT') if 'FT' in u.split(':') else -1)
                for i, u in enumerate(uniq)}
    need_ft = any(v >= 0 for v in ft_index.values())

    gt = np.empty((n_rows, n_samples), dtype=f'U{GT_WIDTH}')
    ft_pass = np.ones((n_rows, n_samples), dtype=bool)
    for j in range(n_samples):
        col = raw[:, j]
        gt[:, j] = np.char.partition(col.astype(f'U{GT_WIDTH}'), ':')[:, 0]
        if not need_ft:
            continue
        wide = col.astype(f'U{CELL_WIDTH}')
        for u_i, k in ft_index.items():
            if k < 0:
                continue
            rows = np.flatnonzero(inv == u_i)
            if rows.size == 0:
                continue
            rest = wide[rows]
            for _ in range(k):
                rest = np.char.partition(rest, ':')[:, 2]
            ft_pass[rows, j] = np.isin(np.char.partition(rest, ':')[:, 0], FT_OK)

    gt = np.char.replace(gt, '|', '/')
    parts = np.char.partition(gt, '/')
    c1, c2 = _codes(parts[:, :, 0]), _codes(parts[:, :, 2])
    first_ok, second_ok = c1 >= 0, c2 >= 0
    alleles = np.full((n_rows, n_samples, 2), -2, dtype=np.int16)
    alleles[..., 0] = np.where(first_ok, c1, np.where(second_ok, c2, -2))
    alleles[..., 1] = np.where(first_ok & second_ok, c2, -2)
    ploidy = first_ok.astype(np.int8) + second_ok.astype(np.int8)
    if need_ft:
        ft_pass |= (ploidy == 0)
    return alleles, ploidy, ft_pass


def annovar_norm(ref, alt):
    """Left- and right-trim a VCF allele pair the way ANNOVAR reports it."""
    i = 0
    while i < len(ref) and i < len(alt) and ref[i] == alt[i]:
        i += 1
    r, a = ref[i:], alt[i:]
    j = 0
    while j < len(r) and j < len(a) and r[len(r) - 1 - j] == a[len(a) - 1 - j]:
        j += 1
    if j:
        r, a = r[:len(r) - j], a[:len(a) - j]
    return (r or '-', a or '-')


def resolve_alt_index(chunk):
    """Index of this row's alternate within the VCF ALT list; NaN when unresolved."""
    n = len(chunk)
    idx = np.ones(n, dtype=np.int16)
    unresolved = np.zeros(n, dtype=bool)
    if 'ALT' not in chunk.columns or 'Alt' not in chunk.columns:
        return idx, unresolved, 0
    alt_series = chunk['ALT'].astype(str)
    vcf_alt = alt_series.to_numpy()
    ann_ref = chunk['Ref'].astype(str).to_numpy()
    ann_alt = chunk['Alt'].astype(str).to_numpy()
    vcf_ref = (chunk['REF'].astype(str).to_numpy() if 'REF' in chunk.columns
               else np.array(['N'] * n))
    multi = np.flatnonzero(alt_series.str.contains(',', regex=False).to_numpy())
    for i in multi:
        options = vcf_alt[i].split(',')
        want = (ann_ref[i], ann_alt[i])
        hit = None
        for k, opt in enumerate(options, start=1):
            if annovar_norm(vcf_ref[i], opt) == want:
                hit = k
                break
        if hit is None:
            unresolved[i] = True          # excluded, never defaulted to 1
        else:
            idx[i] = hit
    return idx, unresolved, len(multi)


def is_present(frame, cols):
    """'this database reports something here'; a frequency of 0 still counts."""
    cols = [c for c in cols if c in frame.columns]
    if not cols:
        return pd.Series(False, index=frame.index)
    out = pd.Series(False, index=frame.index)
    for c in cols:
        v = frame[c].astype('string').str.strip()
        out |= ~(v.isna() | v.isin(MISSING_TOKENS))
    return out


def bin_counts(values, edges):
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v) & (v > 0)]
    return np.histogram(v, bins=edges)[0]


def hwe_chisq_p(n_hom_ref, n_het, n_hom_alt):
    """Vectorised chi-square HWE for the genome-wide flag.

    The exact test (Wigginton 2005) is used for the curated loci in notebook 03;
    at 30 million rows the chi-square approximation is what is affordable, and it
    is only used to raise HWE_extreme at p < 1e-6, never to estimate anything.
    """
    n = n_hom_ref + n_het + n_hom_alt
    with np.errstate(divide='ignore', invalid='ignore'):
        p = (2.0 * n_hom_alt + n_het) / (2.0 * n)
        e_aa = n * p ** 2
        e_ab = 2.0 * n * p * (1.0 - p)
        e_bb = n * (1.0 - p) ** 2
        chi = ((n_hom_alt - e_aa) ** 2 / e_aa
               + (n_het - e_ab) ** 2 / e_ab
               + (n_hom_ref - e_bb) ** 2 / e_bb)
    ok = (n > 0) & np.isfinite(chi) & (e_ab > 0)
    out = np.full(np.shape(chi), np.nan)
    out[ok] = stats.chi2.sf(chi[ok], 1)
    return out


def wilson_lower(k, n, z=1.96):
    """Vectorised Wilson lower bound, used for the enrichment criterion."""
    k = np.asarray(k, dtype=float)
    n = np.asarray(n, dtype=float)
    with np.errstate(divide='ignore', invalid='ignore'):
        p = k / n
        d = 1 + z ** 2 / n
        centre = (p + z ** 2 / (2 * n)) / d
        half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
    return np.where(n > 0, np.maximum(0.0, centre - half), np.nan)


print('parsers, allele resolver and vectorised statistics ready')

## 5. Classification and the single-pass accumulator

`classify_chunk` attaches panel frequencies, the four absence flags, the restriction
cascade, the enrichment flag and the reference-discordance flag.

**Enrichment** now requires three things at once: `AC_pass >= 5`, the cohort frequency
between 1 % and 99 %, and the **Wilson lower bound** of the cohort frequency above
`ENRICHMENT_FOLD` × the reference. The previous point-estimate rule called a site
enriched on five alternate alleles out of 438, where the interval is wider than the
effect. Both `gnomAD_popmax` and `gnomAD_ALL` are carried; popmax is reported.

Correlations use reservoir sampling with a fixed seed and the realised sample size is
reported. No p-values: at two million sites they carry no information.

In [ ]:
class Reservoir:
    """Uniform sample of a stream without holding the stream."""

    def __init__(self, cap, rng):
        self.cap, self.rng = cap, rng
        self.kz = np.empty(cap, dtype=np.float32)
        self.ref = np.empty(cap, dtype=np.float32)
        self.n = 0
        self.filled = 0

    def add(self, kz, ref):
        m = len(kz)
        if m == 0:
            return
        take = min(self.cap - self.filled, m)
        if take:
            self.kz[self.filled:self.filled + take] = kz[:take]
            self.ref[self.filled:self.filled + take] = ref[:take]
            self.filled += take
        if take < m:
            rest_kz, rest_ref = kz[take:], ref[take:]
            offsets = self.n + take + np.arange(len(rest_kz))
            slots = self.rng.integers(0, offsets + 1)
            hit = slots < self.cap
            self.kz[slots[hit]] = rest_kz[hit]
            self.ref[slots[hit]] = rest_ref[hit]
        self.n += m

    def values(self):
        return self.kz[:self.filled], self.ref[:self.filled]


CASCADE = ['polymorphic', 'absent_ruleA', 'chrom_ok', 'index_ok',
           'call_rate_ok', 'snv_only', 'ac_ge_2']
NK_FIELDS = ['count', 'rare', 'common', 'nan_maf', 'snp', 'indel', 'large_indel']


def new_state():
    return {
        'counts': {k: 0 for k in
                   ['total', 'analysable', 'polymorphic', 'reported',
                    'reported_with_catalogue', 'absent_indel', 'absent_singleton',
                    'enriched_popmax', 'enriched_all', 'enriched_pointrule',
                    'discordant', 'fixed', 'hwe_tested', 'hwe_extreme']},
        'cascade': {k: 0 for k in CASCADE},
        'rules': {r: 0 for r in RULES},
        'rules_restricted': {r: 0 for r in RULES},
        'chrom': {},
        'region': {c: 0 for c in REGION_CODES},
        'cross': {},
        'nk': {},
        'nk_tot': {'absent': np.zeros(len(NK_FIELDS), dtype=np.int64),
                   'known': np.zeros(len(NK_FIELDS), dtype=np.int64)},
        'res': {p: Reservoir(CORR_MAX_N, RNG) for p in CORR_PANELS},
        'shared_n': {p: 0 for p in CORR_PANELS},
        'spec_kz': np.zeros(len(KZ_MAF_LABELS), dtype=np.int64),
        'spec_pair': {p: {'kz': np.zeros(len(MAF_LABELS), dtype=np.int64),
                          'ref': np.zeros(len(MAF_LABELS), dtype=np.int64)}
                      for p in SPECTRUM_PANELS},
        'disc': {'func': {}, 'clin': {}, 'exonic': {},
                 'maf': np.zeros(len(DISC_MAF_LABELS), dtype=np.int64),
                 'genes_exonic': set(), 'written': False, 'kept': []},
        'parsing': {'rows': 0, 'multiallelic': 0, 'unresolved': 0,
                    'haploid': 0, 'male_het_x': 0},
    }


def classify_chunk(chunk):
    keep = [c for c in META_COLS if c in chunk.columns]
    slim = chunk[keep].copy()

    kz = pd.to_numeric(slim['AF_pass'], errors='coerce')
    ac = pd.to_numeric(slim['AC_pass'], errors='coerce')
    an = pd.to_numeric(slim['AN_pass'], errors='coerce')
    rate = pd.to_numeric(slim['call_rate_pass'], errors='coerce')
    observed = (kz > 0).fillna(False)

    for rule, cols in RULES.items():
        slim[f'Absent_{rule}'] = observed & ~is_present(slim, cols)
    slim['In_catalogue'] = is_present(slim, CATALOGUE_COLS)

    ref_a = slim['Ref'].astype(str)
    alt_a = slim['Alt'].astype(str)
    dash = (ref_a == '-') | (alt_a == '-')
    slim['is_snv'] = ((ref_a.str.len() == 1) & (alt_a.str.len() == 1) & ~dash)
    chrom = slim['Chr'].astype(str).str.replace('^chr', '', regex=True)
    slim['chrom_ok'] = chrom.isin(ANALYSIS_CHROMS)
    slim['index_ok'] = ~slim['alt_index_unresolved'].astype(bool)
    slim['call_rate_ok'] = (rate >= MIN_CALL_RATE).fillna(False)
    slim['ac_ok'] = (ac >= MIN_AC_REPORTED).fillna(False)

    slim['Reported_absent'] = (slim[f'Absent_{REPORTED_RULE}'] & slim['chrom_ok']
                               & slim['index_ok'] & slim['call_rate_ok']
                               & slim['is_snv'] & slim['ac_ok'])

    for name, src in FREQ_PANELS.items():
        slim[name] = (pd.to_numeric(slim[src], errors='coerce')
                      if src in slim.columns else np.nan)
        if src in slim.columns and src != name:
            slim.drop(columns=[src], inplace=True)

    lower = pd.Series(wilson_lower(ac.to_numpy(), an.to_numpy()), index=slim.index)
    slim['AF_pass_CI_lower'] = lower.round(6)
    base = (slim['chrom_ok'] & slim['index_ok'] & slim['call_rate_ok']
            & (ac >= ENRICH_MIN_AC).fillna(False)
            & (kz > ENRICH_LO) & (kz < ENRICH_HI))
    for field, flag in [('gnomAD_popmax', 'Enriched_popmax'),
                        ('gnomAD_ALL', 'Enriched_ALL')]:
        ref = slim[field] if field in slim.columns else pd.Series(np.nan, index=slim.index)
        slim[flag] = (base & (ref > 0) & (lower > ref * ENRICHMENT_FOLD)).fillna(False)
    ref_pm = slim.get('gnomAD_popmax', pd.Series(np.nan, index=slim.index))
    slim['Enriched_point_rule'] = ((kz > ref_pm * ENRICHMENT_FOLD) & (ref_pm > 0)
                                   & (kz > ENRICH_LO) & (kz < ENRICH_HI)).fillna(False)
    slim['Enriched'] = slim[f'Enriched_{"popmax" if ENRICHMENT_REF.endswith("popmax") else "ALL"}']

    hom = pd.to_numeric(slim['hom_alt_freq_pass'], errors='coerce')
    slim['Ref_discordant'] = ((hom >= DISCORDANT_THRESHOLD) & slim['call_rate_ok']
                              & slim['chrom_ok'] & slim['index_ok']).fillna(False)
    slim['Ref_fixed'] = ((hom >= 1.0) & slim['Ref_discordant']).fillna(False)
    return slim


def _bump(store, key, vec):
    cur = store.get(key)
    if cur is None:
        store[key] = vec.copy()
    else:
        cur += vec


def accumulate(slim, state, write_disc=True):
    c = state['counts']
    kz = pd.to_numeric(slim['AF_pass'], errors='coerce').to_numpy()
    maf = pd.to_numeric(slim['MAF_pass'], errors='coerce').to_numpy()
    ac = pd.to_numeric(slim['AC_pass'], errors='coerce').to_numpy()
    observed = np.isfinite(kz) & (kz > 0)
    chrom_ok = slim['chrom_ok'].to_numpy(dtype=bool)
    index_ok = slim['index_ok'].to_numpy(dtype=bool)
    rate_ok = slim['call_rate_ok'].to_numpy(dtype=bool)
    is_snv = slim['is_snv'].to_numpy(dtype=bool)
    ac_ok = slim['ac_ok'].to_numpy(dtype=bool)
    absent_a = slim[f'Absent_{REPORTED_RULE}'].to_numpy(dtype=bool)
    reported = slim['Reported_absent'].to_numpy(dtype=bool)

    c['total'] += len(slim)
    c['analysable'] += int((chrom_ok & index_ok).sum())
    c['polymorphic'] += int(observed.sum())
    c['reported'] += int(reported.sum())

    step = observed.copy()
    state['cascade']['polymorphic'] += int(step.sum())
    for name, m in (('absent_ruleA', absent_a), ('chrom_ok', chrom_ok),
                    ('index_ok', index_ok), ('call_rate_ok', rate_ok),
                    ('snv_only', is_snv), ('ac_ge_2', ac_ok)):
        step = step & m
        state['cascade'][name] += int(step.sum())

    restrict = chrom_ok & index_ok & rate_ok & is_snv & ac_ok
    for rule in RULES:
        r = slim[f'Absent_{rule}'].to_numpy(dtype=bool)
        state['rules'][rule] += int(r.sum())
        state['rules_restricted'][rule] += int((r & restrict).sum())

    cat = slim['In_catalogue'].to_numpy(dtype=bool)
    c['reported_with_catalogue'] += int((reported & cat).sum())
    c['absent_indel'] += int((absent_a & chrom_ok & index_ok & rate_ok & ~is_snv).sum())
    c['absent_singleton'] += int((absent_a & chrom_ok & index_ok & rate_ok
                                  & is_snv & (ac == 1)).sum())
    for flag, key in (('Enriched_popmax', 'enriched_popmax'),
                      ('Enriched_ALL', 'enriched_all'),
                      ('Enriched_point_rule', 'enriched_pointrule')):
        c[key] += int(slim[flag].to_numpy(dtype=bool).sum())

    disc = slim['Ref_discordant'].to_numpy(dtype=bool)
    c['discordant'] += int(disc.sum())
    c['fixed'] += int(slim['Ref_fixed'].to_numpy(dtype=bool).sum())
    hwe = pd.to_numeric(slim['HWE_p'], errors='coerce').to_numpy()
    hwe_testable = (np.isfinite(hwe) & np.isfinite(maf) & (maf >= HWE_MIN_MAF)
                    & chrom_ok & index_ok & rate_ok)
    c['hwe_tested'] += int(hwe_testable.sum())
    c['hwe_extreme'] += int((hwe_testable & (hwe < HWE_ALPHA)).sum())

    rc = slim['region_class'].to_numpy()
    for code in np.unique(rc):
        state['region'][REGION_CODES[int(code)]] += int((rc == code).sum())

    chrom = slim['Chr'].astype(str).str.replace('^chr', '', regex=True).to_numpy()
    for ch in pd.unique(chrom):
        m = chrom == ch
        _bump(state['chrom'], ch,
              np.array([m.sum(), (m & observed).sum(), (m & reported).sum(),
                        (m & disc).sum()], dtype=np.int64))

    ok = observed & chrom_ok & index_ok & rate_ok & (kz < 1)
    state['spec_kz'] += bin_counts(maf[ok], KZ_MAF_EDGES)
    for panel in CORR_PANELS:
        if panel not in slim.columns:
            continue
        ref = pd.to_numeric(slim[panel], errors='coerce').to_numpy()
        both = ok & np.isfinite(ref) & (ref > 0) & (ref < 1)
        if not both.any():
            continue
        state['shared_n'][panel] += int(both.sum())
        state['res'][panel].add(kz[both].astype(np.float32), ref[both].astype(np.float32))
    for panel in SPECTRUM_PANELS:
        if panel not in slim.columns:
            continue
        ref = pd.to_numeric(slim[panel], errors='coerce').to_numpy()
        both = ok & np.isfinite(ref) & (ref > 0) & (ref < 1)
        if not both.any():
            continue
        state['spec_pair'][panel]['kz'] += bin_counts(maf[both], MAF_EDGES)
        state['spec_pair'][panel]['ref'] += bin_counts(
            np.minimum(ref[both], 1 - ref[both]), MAF_EDGES)

    func = slim['Func.refGene'].astype(str).to_numpy()
    efunc = (slim['ExonicFunc.refGene'].astype(str).to_numpy()
             if 'ExonicFunc.refGene' in slim.columns else np.array(['.'] * len(slim)))
    enr = slim['Enriched'].to_numpy(dtype=bool)
    base = observed & chrom_ok & index_ok & rate_ok
    key_df = pd.DataFrame({'f': func, 'e': efunc,
                           'all': base.astype(np.int64),
                           'abs': reported.astype(np.int64),
                           'enr': (enr & base).astype(np.int64)})
    for (f, e), row in key_df.groupby(['f', 'e'], sort=False)[['all', 'abs', 'enr']] \
            .sum().iterrows():
        _bump(state['cross'], (f, e), row.to_numpy(dtype=np.int64))

    mx = np.maximum(slim['Ref'].astype(str).str.len().to_numpy(),
                    slim['Alt'].astype(str).str.len().to_numpy())
    large = mx > 50
    indel = ~is_snv & ~large
    nk_df = pd.DataFrame({
        'f': func, 'e': efunc,
        'absent': slim[f'Absent_{FILE_SPLIT_RULE}'].to_numpy(dtype=bool),
        'count': base.astype(np.int64),
        'rare': (base & (maf < RARE_MAF)).astype(np.int64),
        'common': (base & (maf >= RARE_MAF)).astype(np.int64),
        'nan_maf': (base & ~np.isfinite(maf)).astype(np.int64),
        'snp': (base & is_snv).astype(np.int64),
        'indel': (base & indel).astype(np.int64),
        'large_indel': (base & large).astype(np.int64)})
    for (f, e, is_abs), row in nk_df.groupby(['f', 'e', 'absent'], sort=False)[
            NK_FIELDS].sum().iterrows():
        cls = 'absent' if is_abs else 'known'
        vec = row.to_numpy(dtype=np.int64)
        _bump(state['nk'], (cls, f, e), vec)
        state['nk_tot'][cls] += vec

    if disc.any():
        sub = slim.loc[disc, [c_ for c_ in DISC_COLS if c_ in slim.columns]].copy()
        d = state['disc']
        for k, v in sub['Func.refGene'].astype(str).value_counts().items():
            d['func'][k] = d['func'].get(k, 0) + int(v)
        ex_mask = sub['Func.refGene'].astype(str).str.startswith('exonic')
        if ex_mask.any() and 'ExonicFunc.refGene' in sub.columns:
            for k, v in sub.loc[ex_mask, 'ExonicFunc.refGene'].astype(str) \
                    .value_counts().items():
                d['exonic'][k] = d['exonic'].get(k, 0) + int(v)
        if 'CLNSIG' in sub.columns:
            cl = sub['CLNSIG'].astype('string').str.strip()
            cl = cl[~(cl.isna() | cl.isin(MISSING_TOKENS))]
            for k, v in cl.value_counts().items():
                d['clin'][k] = d['clin'].get(k, 0) + int(v)
        d['maf'] += bin_counts(pd.to_numeric(sub['MAF_pass'], errors='coerce'),
                               DISC_MAF_EDGES)
        coding = ex_mask & ~sub['Func.refGene'].astype(str).str.contains('ncRNA')
        for field in sub.loc[coding, 'Gene.refGene'].dropna().astype(str):
            for g in field.replace(',', ';').split(';'):
                g = g.strip()
                if g and g not in MISSING_TOKENS and g != 'NONE':
                    d['genes_exonic'].add(g)
        if coding.any():
            d['kept'].append(sub[coding])
        if write_disc:
            sub.to_csv(DISC_FILE, sep='\t', index=False,
                       mode='a' if d['written'] else 'w', header=not d['written'])
            d['written'] = True


def finalize_state(state):
    """Replace the Reservoir objects with plain arrays before caching.

    Pickling a class defined in the notebook ties the cache to this session;
    the sampled values are all that the correlation table needs.
    """
    if 'res' in state:
        state['corr_sample'] = {p: tuple(np.asarray(a) for a in r.values())
                                for p, r in state['res'].items()}
        del state['res']
    return state


print(f'accumulator ready; {len(RULES)} absence rules and a '
      f'{len(CASCADE)}-step restriction cascade per variant')

## 6. Main pass over the callset

One pass computes every table in this notebook. The accumulated state is written to
`{STEM}_scan_state.pkl` with a `.done.json` manifest; if the manifest matches the input
file, its size and the current thresholds, the pass is skipped and everything downstream
rebuilds in seconds. A cache file without its manifest is treated as an interrupted run
and ignored.

In [ ]:
STAGE = 'pass 1 - callset'

MANIFEST_KEY = {
    'input': os.path.basename(VCF),
    'input_size': os.path.getsize(VCF),
    'n_cohort': N_COHORT, 'n_xx': N_XX, 'n_xy': N_XY,
    'min_call_rate': MIN_CALL_RATE, 'min_ac': MIN_AC_REPORTED,
    'enrich': [ENRICHMENT_REF, ENRICHMENT_FOLD, ENRICH_MIN_AC],
    'discordant': DISCORDANT_THRESHOLD,
    'rules': sorted(RULES), 'reported_rule': REPORTED_RULE,
    'analysis_chroms': sorted(ANALYSIS_CHROMS),
    'dry_run': DRY_RUN_ROWS,
    'hwe_min_maf': HWE_MIN_MAF,
}


def cache_valid():
    if not (USE_CACHE and os.path.exists(CACHE_FILE) and os.path.exists(CACHE_DONE)):
        return False
    with open(CACHE_DONE) as fh:
        return json.load(fh).get('key') == MANIFEST_KEY


state = None
if cache_valid():
    with open(CACHE_FILE, 'rb') as fh:
        state = pickle.load(fh)
    register(STAGE, 'input', CACHE_FILE, 'scan state reused')
    print(f'cache hit: {state["parsing"]["rows"]:,} rows, pass skipped')
else:
    if os.path.exists(CACHE_FILE):
        print('cache present but the manifest does not match, re-running')
    for p, note in ((AF_FILE, 'full table') if WRITE_FULL_AF_TABLE else (None, None), \
                    (NOVEL_FILE, f'absent by rule {FILE_SPLIT_RULE}'), \
                    (KNOWN_FILE, 'read by notebook 04'), \
                    (CMP_FILE, 'slim comparison table'), \
                    (DISC_FILE, 'reference-discordant sites')):
        if p is None:
            continue
        if os.path.exists(p) and not OVERWRITE:
            raise FileExistsError(f'{p} exists. Set OVERWRITE = True or move it aside.')
        register(STAGE, 'output', p, note)

    state = new_state()
    written = {k: False for k in ('af', 'absent', 'known', 'cmp')}
    n_absent = n_known = 0

    def _dump(frame, path, key):
        if frame.empty:
            return
        frame.to_csv(path, sep='\t', index=False,
                     mode='a' if written[key] else 'w', header=not written[key])
        written[key] = True

    male = SEX.reindex(sample_cols).to_numpy() == 'XY'

    with tqdm(desc='callset', unit=' rows') as pbar:
        for chunk in pd.read_csv(VCF, sep='\t', chunksize=CHUNKSIZE, low_memory=False):
            alt_idx, unresolved, n_multi = resolve_alt_index(chunk)
            alleles, ploidy, ft_pass = parse_gt_block(chunk[sample_cols],
                                                      chunk['FORMAT'].to_numpy())
            chrom = (chunk['Chr'].astype(str).str.replace('^chr', '', regex=True)
                     .to_numpy())
            rcode = region_code(chrom, pd.to_numeric(chunk['Start'],
                                                     errors='coerce').fillna(0))
            n_ind_exp = _N_IND_VEC[rcode]
            an_exp = _AN_VEC[rcode]

            target = alt_idx[:, None, None]
            called = alleles >= 0
            is_target = called & (alleles == target)
            keep = ft_pass[:, :, None]

            AN_all = called.sum(axis=(1, 2))
            AC_all = is_target.sum(axis=(1, 2))
            AN_pass = (called & keep).sum(axis=(1, 2))
            AC_pass = (is_target & keep).sum(axis=(1, 2))

            n_alt = is_target.sum(axis=2)
            called_s = (ploidy > 0) & ft_pass
            diploid = (ploidy == 2) & ft_pass
            haploid = (ploidy == 1) & ft_pass
            n_hom_alt = ((n_alt == 2) & diploid).sum(axis=1)
            n_het = ((n_alt == 1) & diploid).sum(axis=1)
            n_hemi_alt = ((n_alt == 1) & haploid).sum(axis=1)
            n_called = called_s.sum(axis=1)
            n_hom_ref = n_called - n_het - n_hom_alt - n_hemi_alt

            # a male heterozygous call outside PAR on chrX is a mapping artefact
            x_nonpar = rcode == _CODE['X_nonpar']
            if x_nonpar.any():
                state['parsing']['male_het_x'] += int(
                    (((n_alt == 1) & diploid)[x_nonpar][:, male]).sum())

            with np.errstate(divide='ignore', invalid='ignore'):
                AF_all = np.where(AN_all > 0, AC_all / AN_all, np.nan)
                AF_pass = np.where(AN_pass > 0, AC_pass / AN_pass, np.nan)
                hom_freq = np.where(n_called > 0,
                                    (n_hom_alt + n_hemi_alt) / n_called, np.nan)

            hwe = np.full(len(chunk), np.nan)
            dip = np.isin(rcode, [_CODE['auto'], _CODE['X_par']])
            if dip.any():
                hwe[dip] = hwe_chisq_p(n_hom_ref[dip], n_het[dip], n_hom_alt[dip])

            chunk['AC_all'] = AC_all
            chunk['AN_all'] = AN_all
            chunk['AF_all'] = np.round(AF_all, 6)
            chunk['MAF_all'] = np.round(np.minimum(AF_all, 1 - AF_all), 6)
            chunk['AC_pass'] = AC_pass
            chunk['AN_pass'] = AN_pass
            chunk['AF_pass'] = np.round(AF_pass, 6)
            chunk['MAF_pass'] = np.round(np.minimum(AF_pass, 1 - AF_pass), 6)
            chunk['n_het_pass'] = n_het
            chunk['n_hom_alt_pass'] = n_hom_alt
            chunk['n_hemi_alt_pass'] = n_hemi_alt
            chunk['hom_alt_freq_pass'] = np.round(hom_freq, 4)
            chunk['call_rate_pass'] = np.round(n_called / n_ind_exp, 4)
            chunk['AN_expected'] = an_exp
            chunk['region_class'] = rcode
            chunk['alt_index'] = alt_idx
            chunk['alt_index_unresolved'] = unresolved
            chunk['HWE_p'] = hwe

            p = state['parsing']
            p['rows'] += len(chunk)
            p['multiallelic'] += n_multi
            p['unresolved'] += int(unresolved.sum())
            p['haploid'] += int(haploid.sum())

            slim = classify_chunk(chunk)
            accumulate(slim, state)

            absent_mask = slim[f'Absent_{FILE_SPLIT_RULE}'].to_numpy(dtype=bool)
            n_absent += int(absent_mask.sum())
            n_known += int((~absent_mask).sum())
            if WRITE_FULL_AF_TABLE:
                _dump(chunk, AF_FILE, 'af')
            _dump(chunk[absent_mask], NOVEL_FILE, 'absent')
            _dump(chunk[~absent_mask], KNOWN_FILE, 'known')
            _dump(slim, CMP_FILE, 'cmp')

            pbar.update(len(chunk))
            del chunk, slim, alleles, called, is_target
            if DRY_RUN_ROWS and state['parsing']['rows'] >= DRY_RUN_ROWS:
                print(f'\ndry run stopped at {state["parsing"]["rows"]:,} rows')
                break
            gc.collect()

    assert n_absent + n_known == state['parsing']['rows'], \
        'the two files must partition the callset exactly'
    state['file_split'] = {'absent': n_absent, 'known': n_known}
    finalize_state(state)

    if not DRY_RUN_ROWS:
        with open(CACHE_FILE, 'wb') as fh:
            pickle.dump(state, fh, protocol=4)
        with open(CACHE_DONE, 'w') as fh:
            json.dump({'key': MANIFEST_KEY,
                       'written': datetime.now().isoformat(timespec='seconds'),
                       'rows': state['parsing']['rows']}, fh, indent=1)
        register(STAGE, 'output', CACHE_FILE, 'scan state')

par = state['parsing']
parsing_df = pd.DataFrame([
    ('Rows in the annotated callset', par['rows']),
    ('Rows from multiallelic sites', par['multiallelic']),
    ('Rows excluded, alternate index unresolved', par['unresolved']),
    ('Haploid genotype calls', par['haploid']),
    ('Heterozygous calls in XY individuals on chrX outside PAR', par['male_het_x']),
    ('Rows on chrY outside PAR, excluded from allele frequencies',
     state['region']['Y_nonpar']),
    ('Rows on chrX outside PAR', state['region']['X_nonpar']),
    ('Rows in PAR1 or PAR2', state['region']['X_par'] + state['region']['Y_par']),
], columns=['Item', 'Count'])
print(f'rows processed: {par["rows"]:,}')
display(parsing_df)

## 7. Tables

**Restriction cascade.** Every step from "polymorphic in the cohort" to the reported set
is counted, so the effect of each restriction is visible rather than asserted.

**Absence rules.** All four rules are reported both unrestricted and under the cascade.

**Functional composition.** log2 fold changes with 95 % confidence intervals, Haldane 0.5
correction, and a Benjamini-Hochberg q-value across categories.

**Reference-discordant sites.** Positions where at least 80 % of PASS-genotyped
individuals carry the alternate allele homozygously or hemizygously, at call rate ≥ 90 %,
on autosomes and chrX with a resolved allele index.

In [ ]:
counts = state['counts']
cascade = state['cascade']
par = state['parsing']
obs = counts['polymorphic']
n_reported = counts['reported']
analysable = counts['analysable']

cascade_df = pd.DataFrame([
    ('Polymorphic in the cohort', cascade['polymorphic'], ''),
    (f'Absent under rule {REPORTED_RULE}', cascade['absent_ruleA'],
     RULE_TEXT[REPORTED_RULE]),
    ('On an autosome or chrX', cascade['chrom_ok'], 'chrY carries no comparable AF'),
    ('Alternate allele index resolved', cascade['index_ok'], ''),
    (f'Call rate >= {MIN_CALL_RATE:.0%} of expected ploidy', cascade['call_rate_ok'], ''),
    ('Single-nucleotide substitution', cascade['snv_only'],
     'indel representation does not match the panels'),
    (f'AC >= {MIN_AC_REPORTED}', cascade['ac_ge_2'], 'singletons excluded'),
], columns=['Step', 'Variants remaining', 'Note'])
cascade_df['Percent of polymorphic'] = (
    cascade_df['Variants remaining'] / max(obs, 1) * 100).round(3)
cascade_df['Removed by this step'] = (
    cascade_df['Variants remaining'].shift(1) - cascade_df['Variants remaining']).fillna(0).astype(int)

DEN = {'annotated': counts['total'], 'polymorphic': obs, 'reported': n_reported}

def _r(q, d, n, den_name):
    return {'Quantity or class': q, 'Definition': d, 'n': int(n),
            '% of denominator': round(n / max(DEN[den_name], 1) * 100, 3),
            'Denominator': den_name}

def _h(t):
    return {'Quantity or class': t, 'Definition': '', 'n': None,
            '% of denominator': None, 'Denominator': ''}

n_residual = n_reported - counts['reported_with_catalogue']

summary_df = pd.DataFrame([
    _h('Callset composition'),
    _r('Annotated variant records', 'ANNOVAR records, one per alternate allele',
       counts['total'], 'annotated'),
    _r('  from multiallelic positions', '', par['multiallelic'], 'annotated'),
    _r('  alternate index unresolvable', 'Excluded from allele counting',
       par['unresolved'], 'annotated'),
    _r('Analysable', 'Autosome or chromosome X, allele index resolved',
       analysable, 'annotated'),
    _r('Polymorphic in the cohort', 'AF > 0', obs, 'annotated'),

    _h('Restriction cascade, applied in order to the polymorphic records'),
    _r(f'Absent under rule {REPORTED_RULE}', RULE_TEXT[REPORTED_RULE],
       cascade['absent_ruleA'], 'polymorphic'),
    _r('  on an autosome or chromosome X',
       f"Removes {cascade['absent_ruleA'] - cascade['chrom_ok']:,}",
       cascade['chrom_ok'], 'polymorphic'),
    _r('  alternate allele index resolved',
       f"Removes {cascade['chrom_ok'] - cascade['index_ok']:,}",
       cascade['index_ok'], 'polymorphic'),
    _r(f'  call rate >= {MIN_CALL_RATE:.0%} of expected ploidy',
       f"Removes {cascade['index_ok'] - cascade['call_rate_ok']:,}",
       cascade['call_rate_ok'], 'polymorphic'),
    _r('  single-nucleotide substitution',
       f"Removes {counts['absent_indel']:,} indels", cascade['snv_only'], 'polymorphic'),
    _r(f'  allele count >= {MIN_AC_REPORTED}, the reported set',
       f"Removes {counts['absent_singleton']:,} singletons",
       cascade['ac_ge_2'], 'polymorphic'),

    _h('The reported set partitioned by catalogue support'),
    _r('  with a dbSNP, COSMIC or ClinVar record',
       'Present in a catalogue without a frequency',
       counts['reported_with_catalogue'], 'reported'),
    _r('  with no catalogue record, the residual set', '',
       n_residual, 'reported'),

    _h('Classes defined independently of the cascade'),
    _r(f'Enriched against {ENRICHMENT_REF}',
       f'Wilson lower bound above {ENRICHMENT_FOLD:.0f} x reference, AC >= {ENRICH_MIN_AC}',
       counts['enriched_popmax'], 'polymorphic'),
    _r('Enriched against gnomAD ALL', 'Same rule, aggregate denominator',
       counts['enriched_all'], 'polymorphic'),
    _r('Enriched under the previous point-estimate rule', 'Reported for comparison only',
       counts['enriched_pointrule'], 'polymorphic'),
    _r('Reference-discordant',
       f'Homozygous-alternate frequency >= {DISCORDANT_THRESHOLD:.0%}, '
       f'call rate >= {MIN_CALL_RATE:.0%}', counts['discordant'], 'polymorphic'),
    _r('  fully fixed', 'No reference allele observed in the cohort',
       counts['fixed'], 'polymorphic'),
    _r(f'Tested for Hardy-Weinberg equilibrium (MAF >= {HWE_MIN_MAF:.0%})',
       'Diploid regions only', counts['hwe_tested'], 'polymorphic'),
    _r(f'  departing at P < {HWE_ALPHA:g}', 'Exact test',
       counts['hwe_extreme'], 'polymorphic'),
])

rules_df = pd.DataFrame([
    {'Rule': r, 'Databases': RULE_TEXT[r],
     'Frequency panels only': not any(c in RULES[r] for c in CATALOGUE_COLS),
     'Absent, unrestricted': state['rules'][r],
     'Absent, restricted set': state['rules_restricted'][r],
     'Restricted, % of polymorphic':
         round(state['rules_restricted'][r] / max(obs, 1) * 100, 3),
     'Role': ('reported' if r == REPORTED_RULE else
              'defines the file split' if r == FILE_SPLIT_RULE else 'sensitivity')}
    for r in RULES])

corr_rows = []
for panel in CORR_PANELS:
    a, b = state['corr_sample'].get(panel, (np.array([]), np.array([])))
    if len(a) < 100:
        continue
    r, _ = stats.spearmanr(a, b)
    corr_rows.append({'Panel': panel, 'Spearman r': round(float(r), 4),
                      'Shared polymorphic sites': state['shared_n'][panel],
                      'Sites used': len(a)})
corr_df = (pd.DataFrame(corr_rows).sort_values('Spearman r', ascending=False)
           .reset_index(drop=True))

spec_kz_df = pd.DataFrame({
    'MAF bin': KZ_MAF_LABELS, 'Variants': state['spec_kz'],
    'Percent': np.round(state['spec_kz'] / max(state['spec_kz'].sum(), 1) * 100, 2)})

spec_rows = []
for panel in SPECTRUM_PANELS:
    kzv = state['spec_pair'][panel]['kz']
    refv = state['spec_pair'][panel]['ref']
    kzp = kzv / max(kzv.sum(), 1) * 100
    refp = refv / max(refv.sum(), 1) * 100
    for i, lab in enumerate(MAF_LABELS):
        spec_rows.append({'Reference panel': panel, 'MAF bin': lab,
                          'Shared sites': int(kzv[i]),
                          f'{COHORT} (%)': round(float(kzp[i]), 2),
                          'Reference (%)': round(float(refp[i]), 2),
                          'Ratio': round(float(kzp[i] / refp[i]) if refp[i] else np.nan, 3)})
spec_df = pd.DataFrame(spec_rows)

FUNC_ORDER = [
    ('Non-coding, large regions', ['intergenic', 'intronic', 'ncRNA_intronic']),
    ('Regulatory regions', ['UTR3', 'UTR5', 'UTR5;UTR3', 'upstream', 'downstream',
                            'upstream;downstream']),
    ('Non-coding RNA', ['ncRNA_exonic', 'ncRNA_splicing', 'ncRNA_exonic;splicing']),
    ('Splicing', ['splicing', 'exonic;splicing']),
    ('Protein coding, exonic', ['exonic']),
]
EXONIC_ORDER = ['synonymous SNV', 'nonsynonymous SNV', 'stopgain', 'stoploss',
                'startloss', 'frameshift insertion', 'frameshift deletion',
                'nonframeshift insertion', 'nonframeshift deletion', 'unknown']
FUNC_LABEL = {'intergenic': 'Intergenic', 'intronic': 'Intronic',
              'ncRNA_intronic': 'ncRNA intronic', 'UTR3': "3' UTR", 'UTR5': "5' UTR",
              'UTR5;UTR3': "5'/3' UTR overlap", 'upstream': 'Upstream',
              'downstream': 'Downstream',
              'upstream;downstream': 'Upstream/downstream overlap',
              'ncRNA_exonic': 'ncRNA exonic', 'ncRNA_splicing': 'ncRNA splicing',
              'ncRNA_exonic;splicing': 'ncRNA exonic-splicing',
              'splicing': 'Splicing', 'exonic;splicing': 'Exonic-splicing',
              'exonic': 'Exonic'}

func_all, func_abs, func_enr = {}, {}, {}
ex_all, ex_abs, ex_enr = {}, {}, {}
for (f, e), v in state['cross'].items():
    for store, i in ((func_all, 0), (func_abs, 1), (func_enr, 2)):
        store[f] = store.get(f, 0) + int(v[i])
    if f.startswith('exonic'):
        for store, i in ((ex_all, 0), (ex_abs, 1), (ex_enr, 2)):
            store[e] = store.get(e, 0) + int(v[i])

tot = {'all': sum(func_all.values()), 'absent': sum(func_abs.values()),
       'enriched': sum(func_enr.values())}
assert tot['absent'] == n_reported, 'cross-tab and classification disagree'

rows = []
for section, keys in FUNC_ORDER:
    for key in keys:
        n_a = func_all.get(key, 0)
        if n_a == 0:
            continue
        n_p, n_e = func_abs.get(key, 0), func_enr.get(key, 0)
        fc, lo, hi = log2fc_ci(n_p, n_a - n_p, tot['absent'], tot['all'] - tot['absent'])
        rows.append({'Section': section, 'Category': FUNC_LABEL.get(key, key),
                     'Subtype': '', 'ANNOVAR key': key,
                     'Analysable variants': n_a,
                     'Analysable (%)': round(n_a / max(tot['all'], 1) * 100, 3),
                     'Absent': n_p,
                     'Absent (%)': round(n_p / max(tot['absent'], 1) * 100, 3),
                     'Enriched': n_e,
                     'log2FC': round(fc, 3), 'CI lower': round(lo, 3),
                     'CI upper': round(hi, 3)})
for key in EXONIC_ORDER:
    n_a = ex_all.get(key, 0)
    if n_a == 0:
        continue
    n_p, n_e = ex_abs.get(key, 0), ex_enr.get(key, 0)
    fc, lo, hi = log2fc_ci(n_p, n_a - n_p, tot['absent'], tot['all'] - tot['absent'])
    rows.append({'Section': 'Protein coding, exonic subtype', 'Category': 'Exonic',
                 'Subtype': key[0].upper() + key[1:], 'ANNOVAR key': key,
                 'Analysable variants': n_a,
                 'Analysable (%)': round(n_a / max(tot['all'], 1) * 100, 4),
                 'Absent': n_p,
                 'Absent (%)': round(n_p / max(tot['absent'], 1) * 100, 4),
                 'Enriched': n_e,
                 'log2FC': round(fc, 3), 'CI lower': round(lo, 3),
                 'CI upper': round(hi, 3)})
func_table = pd.DataFrame(rows)
z = (func_table['log2FC'] / ((func_table['CI upper'] - func_table['CI lower']) / 3.92)
     .replace(0, np.nan))
func_table['q (BH)'] = np.round(bh(2 * stats.norm.sf(np.abs(z.to_numpy()))), 4)
func_table['CI excludes zero'] = (func_table['CI lower'] * func_table['CI upper']) > 0

nk_rows = {}
for (cls, f, e), v in state['nk'].items():
    r = nk_rows.setdefault((f, e), {'Func': f, 'ExonicFunc': e})
    for i, name in enumerate(NK_FIELDS):
        r[f'{cls}_{name}'] = int(v[i])
nk_full = pd.DataFrame(nk_rows.values()).fillna(0)
for cls in ('absent', 'known'):
    for name in NK_FIELDS:
        col = f'{cls}_{name}'
        if col not in nk_full.columns:
            nk_full[col] = 0
        nk_full[col] = nk_full[col].astype(int)
nk_full['total'] = nk_full['absent_count'] + nk_full['known_count']
nk_full['absent_to_known'] = (nk_full['absent_count']
                              / nk_full['known_count'].replace(0, np.nan))
T_ABS = int(state['nk_tot']['absent'][0])
T_KNO = int(state['nk_tot']['known'][0])
_ci = [log2fc_ci(r.absent_count, r.known_count, T_ABS, T_KNO)
       for r in nk_full.itertuples()]
nk_full['log2FC'] = [x[0] for x in _ci]
nk_full['CI lower'] = [x[1] for x in _ci]
nk_full['CI upper'] = [x[2] for x in _ci]
nk_full = nk_full.sort_values('total', ascending=False).reset_index(drop=True)

indel_check = pd.DataFrame([
    {'Class': lab,
     'Absent': int(nk_full.loc[nk_full['ExonicFunc'] == key, 'absent_count'].sum()),
     'Known': int(nk_full.loc[nk_full['ExonicFunc'] == key, 'known_count'].sum())}
    for key, lab in (('synonymous SNV', 'Synonymous SNV'),
                     ('nonsynonymous SNV', 'Nonsynonymous SNV'),
                     ('frameshift insertion', 'Frameshift insertion'),
                     ('frameshift deletion', 'Frameshift deletion'),
                     ('nonframeshift insertion', 'Nonframeshift insertion'),
                     ('nonframeshift deletion', 'Nonframeshift deletion'))])
indel_check['Absent/Known'] = (indel_check['Absent']
                               / indel_check['Known'].replace(0, np.nan)).round(4)

d = state['disc']
disc_func_df = pd.DataFrame(sorted(d['func'].items(), key=lambda x: -x[1]),
                            columns=['Func.refGene', 'Sites'])
if not disc_func_df.empty:
    disc_func_df['Percent'] = (disc_func_df['Sites']
                               / max(counts['discordant'], 1) * 100).round(2)
disc_maf_df = pd.DataFrame({'MAF bin': DISC_MAF_LABELS, 'Sites': d['maf'],
                            'Percent': np.round(d['maf'] / max(d['maf'].sum(), 1)
                                                * 100, 2)})


def clinvar_bucket(text):
    t = str(text).lower()
    if 'conflicting' in t:
        return 'Conflicting'
    if 'pathogenic' in t and 'benign' not in t:
        return 'Pathogenic or likely pathogenic'
    if 'benign' in t:
        return 'Benign or likely benign'
    if 'uncertain' in t:
        return 'Uncertain significance'
    return 'Other'


clin_rows = {}
for k, v in d['clin'].items():
    b = clinvar_bucket(k)
    clin_rows[b] = clin_rows.get(b, 0) + v
disc_clin_df = pd.DataFrame(sorted(clin_rows.items(), key=lambda x: -x[1]),
                            columns=['ClinVar class', 'Sites'])
if not disc_clin_df.empty:
    disc_clin_df['Percent of annotated'] = (
        disc_clin_df['Sites'] / disc_clin_df['Sites'].sum() * 100).round(2)
disc_coding_df = (pd.concat(d['kept'], ignore_index=True) if d['kept']
                  else pd.DataFrame(columns=DISC_COLS))
with open(GENES_EXONIC, 'w') as fh:
    fh.write('\n'.join(sorted(d['genes_exonic'])) + '\n')
register('reference-discordant', 'output', GENES_EXONIC, 'exonic genes only')

chrom_df = pd.DataFrame(
    [{'Chromosome': ch, 'Length (Mb)': HG19_LEN.get(ch, np.nan) / 1e6,
      'Callset rows': int(v[0]), 'Polymorphic': int(v[1]),
      'Absent, reported set': int(v[2]), 'Reference-discordant': int(v[3])}
     for ch, v in state['chrom'].items()])
chrom_df = chrom_df[chrom_df['Chromosome'].isin(CHROM_ORDER)].copy()
chrom_df['Chromosome'] = pd.Categorical(chrom_df['Chromosome'],
                                        categories=CHROM_ORDER, ordered=True)
chrom_df = chrom_df.sort_values('Chromosome').reset_index(drop=True)
chrom_df['In analysis'] = chrom_df['Chromosome'].astype(str).isin(ANALYSIS_CHROMS)
for src, dst in (('Polymorphic', 'Polymorphic per Mb'),
                 ('Absent, reported set', 'Absent per Mb'),
                 ('Reference-discordant', 'Reference-discordant per Mb')):
    chrom_df[dst] = (chrom_df[src] / chrom_df['Length (Mb)']).round(2)
chrom_df['Length (Mb)'] = chrom_df['Length (Mb)'].round(1)

print(f'analysable variants       : {analysable:,}')
print(f'polymorphic               : {obs:,}')
print(f'reported absent set       : {n_reported:,} '
      f'({n_reported / max(obs, 1) * 100:.2f}% of polymorphic)')
print(f'unrestricted rule A       : {state["rules"][REPORTED_RULE]:,} '
      f'({state["rules"][REPORTED_RULE] / max(obs, 1) * 100:.2f}%, upper bound)')
display(cascade_df)
display(rules_df)
display(indel_check)
display(summary_df)

In [ ]:
# residual set: absent and uncatalogued

RESIDUAL_COLS = ['Chr', 'Start', 'Ref', 'Alt', 'Func.refGene', 'Gene.refGene',
                 'ExonicFunc.refGene', 'AF_pass', 'AC_pass', 'AN_pass', 'MAF_pass',
                 'call_rate_pass']

print(f'reading {CMP_FILE}')
residual_parts, kept_parts = [], []
n_seen = 0
for slim in pd.read_csv(CMP_FILE, sep='\t', chunksize=CHUNKSIZE, low_memory=False):
    rep = slim['Reported_absent'].to_numpy(dtype=bool)
    cat = slim['In_catalogue'].to_numpy(dtype=bool)
    cols = [c for c in RESIDUAL_COLS if c in slim.columns]
    if rep.any():
        residual_parts.append(slim.loc[rep & ~cat, cols].copy())
        kept_parts.append(slim.loc[rep & cat, cols].copy())
    n_seen += len(slim)
    del slim
    gc.collect()

df_residual = pd.concat(residual_parts, ignore_index=True) if residual_parts else pd.DataFrame()
df_catalogued = pd.concat(kept_parts, ignore_index=True) if kept_parts else pd.DataFrame()
del residual_parts, kept_parts
gc.collect()

n_res, n_cat = len(df_residual), len(df_catalogued)
print(f'rows read {n_seen:,}')
print(f'  absent from every panel, AC >= 2, SNV, call rate ok : {n_res + n_cat:,}')
print(f'    with a dbSNP, COSMIC or ClinVar record            : {n_cat:,}')
print(f'    with no catalogue record, the residual set        : {n_res:,}')
assert n_res + n_cat == state['cascade']['ac_ge_2'], (
    f"second pass gives {n_res + n_cat}, the cascade counted "
    f"{state['cascade']['ac_ge_2']}; the masks have diverged")
print('  reconciles with the cascade count')

In [ ]:
assert n_cat == state['counts']['reported_with_catalogue'], (
    f"catalogued split gives {n_cat}, pass 1 counted "
    f"{state['counts']['reported_with_catalogue']}")
print('  catalogue split reconciles too')

In [ ]:
from scipy.stats import fisher_exact

TI = {('A','G'), ('G','A'), ('C','T'), ('T','C')}
COMP = {'A':'T','T':'A','G':'C','C':'G'}

def _subs(d):
    r = d['Ref'].astype(str).str.upper()
    a = d['Alt'].astype(str).str.upper()
    ok = r.str.len().eq(1) & a.str.len().eq(1)
    return list(zip(r[ok], a[ok]))

def titv(d):
    p = _subs(d)
    ti = sum(1 for x in p if x in TI)
    return (ti / (len(p) - ti) if len(p) - ti else np.nan), len(p)

def spectrum(d):
    fold = lambda p: f'{p[0]}>{p[1]}' if p[0] in 'CT' else f'{COMP[p[0]]}>{COMP[p[1]]}'
    return pd.Series([fold(p) for p in _subs(d)]).value_counts(normalize=True).sort_index() * 100

r_res, n_res_sub = titv(df_residual)
r_cat, n_cat_sub = titv(df_catalogued)
s_res, s_cat = spectrum(df_residual), spectrum(df_catalogued)

titv_df = pd.DataFrame([
    {'Set': 'absent, with a catalogue record', 'Substitutions': n_cat_sub, 'Ti/Tv': round(r_cat, 3)},
    {'Set': 'absent, uncatalogued',            'Substitutions': n_res_sub, 'Ti/Tv': round(r_res, 3)},
])

spec_df_cmp = pd.DataFrame({
    'Catalogued %': s_cat.round(2),
    'Uncatalogued %': s_res.round(2),
    'Difference (pp)': (s_res - s_cat).round(2),
}).rename_axis('Substitution class').reset_index()

d = df_residual.copy()
for c in ('Start', 'AF_pass', 'AC_pass', 'call_rate_pass'):
    if c in d:
        d[c] = pd.to_numeric(d[c], errors='coerce')
d = d.sort_values(['Chr', 'Start'])
gap = np.fmin(d.groupby('Chr')['Start'].diff().fillna(np.inf),
              d.groupby('Chr')['Start'].diff(-1).abs().fillna(np.inf))
hwe = pd.to_numeric(d['hwe_p'], errors='coerce') if 'hwe_p' in d else pd.Series(1.0, index=d.index)

rows, keep = [], pd.Series(True, index=d.index)
for lab, cond in [('none', pd.Series(True, index=d.index)),
                  ('call rate >= 0.98', d['call_rate_pass'] >= 0.98),
                  ('AC >= 3',           d['AC_pass'] >= 3),
                  ('no neighbour within 100 bp', gap > 100),
                  ('HWE p >= 1e-6',     hwe >= 1e-6)]:
    keep &= cond.fillna(False)
    rr, nn = titv(d[keep])
    rows.append({'Cumulative filter': lab, 'Variants': nn, 'Ti/Tv': round(rr, 3)})
artefact_df = pd.DataFrame(rows)

af_r = pd.to_numeric(df_residual['AF_pass'], errors='coerce').dropna()
af_c = pd.to_numeric(df_catalogued['AF_pass'], errors='coerce').dropna()
b_r = int(((af_r >= .45) & (af_r <= .55)).sum())
b_c = int(((af_c >= .45) & (af_c <= .55)).sum())
odds, p_band = fisher_exact([[b_r, len(af_r) - b_r], [b_c, len(af_c) - b_c]])

print(titv_df.to_string(index=False))
print()
print(spec_df_cmp.to_string(index=False))
print()
print(artefact_df.to_string(index=False))
print(f'\n0.45-0.55 allele-frequency band: uncatalogued {b_r / len(af_r) * 100:.2f}%, '
      f'catalogued {b_c / len(af_c) * 100:.2f}%, OR {odds:.2f}, p = {p_band:.3g}')
print(f'\nC>T difference {s_res.get("C>T", 0) - s_cat.get("C>T", 0):+.2f} pp, '
      f'T>C difference {s_res.get("T>C", 0) - s_cat.get("T>C", 0):+.2f} pp')

In [ ]:
OUT_TSV = os.path.join(OUT_DIR, 'KZ219_absent_uncatalogued.tsv.gz')
df_residual.to_csv(OUT_TSV, sep='\t', index=False, compression='gzip')

OUT_PARQUET = os.path.join(OUT_DIR, 'KZ219_absent_uncatalogued.parquet')
df_residual.to_parquet(OUT_PARQUET, index=False, compression='zstd')

OUT_BED = os.path.join(OUT_DIR, 'KZ219_absent_uncatalogued.bed.gz')
bed = pd.DataFrame({
    'chrom': df_residual['Chr'].astype(str),
    'start': pd.to_numeric(df_residual['Start'], errors='coerce').astype('Int64') - 1,
    'end':   pd.to_numeric(df_residual['Start'], errors='coerce').astype('Int64'),
    'name':  df_residual['Ref'].astype(str) + '/' + df_residual['Alt'].astype(str),
    'score': pd.to_numeric(df_residual['AC_pass'], errors='coerce').astype('Int64'),
}).sort_values(['chrom', 'start'])
bed.to_csv(OUT_BED, sep='\t', index=False, header=False, compression='gzip')

for p in (OUT_TSV, OUT_PARQUET, OUT_BED):
    print(f'saved {os.path.basename(p):44s} {os.path.getsize(p)/1e6:7.1f} MB')
print(f'{len(df_residual):,} variants, Ti/Tv {r_res:.3f}')

## 8. Supplementary Fig. 6 - how the reported count was obtained

This figure exists so that the restriction cascade and the rule-to-rule disagreement are
in the paper rather than in a reviewer's question. **a** the cascade. **b** the four
absence rules, unrestricted and restricted (hatched). **c** absent-to-known ratio by variant
class (indels hatched), which is why indels are not in the headline count.

In [ ]:
from matplotlib.ticker import MaxNLocator
INK, RULE, FAINT, BAND = '#1A1A1A', '#3C3C3C', '#B0B0B0', '#4A4A4A'
REF_BLUE, REF_BRICK, NEUTRAL = '#2C6E9B', '#B5502F', '#A0A0A0'
COL.update({'absent': REF_BRICK, 'shared': REF_BLUE, 'neutral': NEUTRAL,
            'ci': RULE, 'link': '#C8CFD6', 'caption': BAND})
PT = FS
BAR_EDGE = {'edgecolor': 'white', 'linewidth': 0.4}

fig, axs = plt.subplots(1, 3, figsize=(FIG_W_MM * MM, 92 * MM),
                        gridspec_kw={'width_ratios': [1.35, 0.95, 1.0]})
LABEL_BOX = dict(boxstyle='square,pad=0.1', fc='white', ec='none')   # value labels sit over grid and reference lines

# a, restriction cascade
ax = axs[0]
cd = cascade_df.copy()
cd['Step'] = (cd['Step'].str.replace(r'^Absent under rule (\w)_.*$', r'Absent under rule \1',
                                     regex=True)
                .str.replace('>=', '\u2265', regex=False)
                .str.replace('Call rate \u2265 90% of expected ploidy',
                             'Call rate \u2265 90% of\nexpected ploidy', regex=False)
                .str.replace('Alternate allele index resolved',
                             'Alternate allele\nindex resolved', regex=False)
                .str.replace('Single-nucleotide substitution',
                             'Single-nucleotide\nsubstitution', regex=False))
v = cd['Variants remaining'].to_numpy(dtype=float)
y = np.arange(len(cd))[::-1]
base = v.min() / 2.2
ax.hlines(y, base, v, color=COL['link'], lw=1.0, zorder=1)
ax.scatter(v, y, s=28, color=COL['absent'], edgecolor='white', lw=0.4, zorder=3)
for yi, (val, rem) in enumerate(zip(v, cd['Removed by this step'])):
    txt = f'{val/1e6:.2f}M' if val >= 1e6 else f'{val:,.0f}'
    if yi:                                  # what this step took out
        txt += f'   \u2212{rem / v[yi - 1] * 100:.0f}%'
    ax.annotate(txt, (val, y[yi]), textcoords='offset points', xytext=(6, 0),
                va='center', ha='left', fontsize=PT, bbox=LABEL_BOX, zorder=5)
ax.set_xscale('log')
ax.set_xlim(base, v.max() * 12)
ax.set_yticks(y)
ax.set_yticklabels(cd['Step'], fontsize=PT)
ax.set_ylim(-0.6, len(cd) - 0.4)
ax.set_xlabel('Variants remaining (log scale)')
ax.grid(True, axis='x')
ax.tick_params(axis='y', length=0)
ax.spines['left'].set_visible(False)

# b, absence rules 
ax = axs[1]
x = np.arange(len(rules_df))
w = 0.38
unr = rules_df['Absent, unrestricted'].to_numpy(dtype=float) / 1e6
res = rules_df['Absent, restricted set'].to_numpy(dtype=float) / 1e6
ax.bar(x - w / 2, unr, w, color=COL['neutral'], label='unrestricted', **BAR_EDGE)
ax.bar(x + w / 2, res, w, color=COL['absent'], label='restricted', hatch='////', **BAR_EDGE)
for xi, (u, r) in enumerate(zip(unr, res)):          # one label per pair, above the pair
    ax.annotate(f'{r / u * 100:.0f}%', (xi, u), textcoords='offset points',
                xytext=(0, 3), ha='center', va='bottom', fontsize=PT, color=COL['caption'],
                bbox=LABEL_BOX, zorder=5)
ax.set_xticks(x)
ax.set_xticklabels([r.split('_')[0] for r in rules_df['Rule']], fontsize=PT)
ax.set_ylim(0, unr.max() * 1.50)        # head-room for the 8 pt legend
ax.set_ylabel('Variants (millions)')        # explicit, not a stray 1e6 offset label
ax.set_xlabel('Absence rule')
ax.legend(fontsize=PT, handlelength=1.1, handletextpad=0.4, loc='upper left',
          borderaxespad=0.2)
ax.grid(True, axis='y')

# c, absent-to-known ratio by class
ax = axs[2]
ic = indel_check.dropna(subset=['Absent/Known']).sort_values('Absent/Known')
is_snv = ic['Class'].str.contains('SNV')
y = np.arange(len(ic))
ax.barh(y, ic['Absent/Known'],
        color=[COL['shared'] if s else COL['absent'] for s in is_snv],
        hatch=[None if s else '////' for s in is_snv],
        height=0.66, zorder=3, **BAR_EDGE)
snv_max = float(ic.loc[is_snv, 'Absent/Known'].max()) if is_snv.any() else np.nan
if np.isfinite(snv_max):
    ax.axvline(snv_max, color=COL['ci'], lw=0.6, ls='--', zorder=2)
for yi, val in enumerate(ic['Absent/Known']):
    ax.annotate(f'{val:.2f}', (val, yi), textcoords='offset points', xytext=(3, 0),
                va='center', ha='left', fontsize=PT, color=COL['caption'],
                bbox=LABEL_BOX, zorder=5)
ax.set_yticks(y)
ax.set_yticklabels(ic['Class'].str.replace(' ', '\n', n=1), fontsize=PT, linespacing=1.0)
ax.set_ylim(-0.5, len(ic) - 0.5)
ax.set_xlim(0, float(ic['Absent/Known'].max()) * 1.55)
ax.xaxis.set_major_locator(MaxNLocator(3))
ax.set_xlabel('Absent-to-known\nratio')
ax.grid(True, axis='x')
ax.tick_params(axis='y', length=0)
ax.spines['left'].set_visible(False)
ax.legend([plt.Rectangle((0, 0), 1, 1, facecolor=COL['shared'], edgecolor='white'),
           plt.Rectangle((0, 0), 1, 1, facecolor=COL['absent'], edgecolor='white',
                         hatch='////')],
          ['SNV', 'indel'], fontsize=PT, loc='lower right', handlelength=1.0,
          handletextpad=0.4, borderaxespad=0.2, framealpha=1.0, facecolor='white',
          edgecolor='none')

fig.tight_layout(rect=[0, 0, 1, 0.93], pad=0.6, w_pad=2.0)
label_panels(fig, axs)
FIG_STEM = f"Supplementary_Fig_{SUPP_NUM['fig_absence_sensitivity']}_absence_sensitivity"
check_min_font(fig, label=FIG_STEM)
save_fig(fig, FIG_STEM, OUT_DIR)
plt.show()

print(f"Supplementary Fig. {SUPP_NUM['fig_absence_sensitivity']}. Sensitivity of the count of "
      "population-restricted variants to its definition. a, Restriction cascade on a "
      "logarithmic axis: number of variants remaining after each step, with the percentage "
      "removed by that step. b, Variants absent from the queried panels under each of the four "
      "absence rules before restriction (grey) and after the full cascade (orange, hatched); "
      "the percentage above each pair is the restricted count as a share of the unrestricted "
      "count. c, Ratio of absent to known records by exonic class; single-nucleotide "
      "substitutions in blue, insertions and deletions in orange, hatched; the dashed line "
      "marks the highest ratio among single-nucleotide classes. Source data are provided in "
      f"Supplementary Data {SUPP_NUM['data']}, sheets 03_RestrictionCascade, 02_absence_rules "
      "and 05_indel_sensitivity.")

## 9. Session record and the values for the Results text

`SessionInfo` records software versions and every threshold; file paths are recorded as
basenames so the deposited workbook carries no directory structure.

In [ ]:
session = pd.DataFrame([
    ('generated', datetime.now().strftime('%Y-%m-%d %H:%M')),
    ('cohort', f'{COHORT}, n = {N_COHORT} ({N_XX} XX, {N_XY} XY)'),
    ('input', os.path.basename(VCF)),
    ('reference', 'GRCh37/hg19'),
    ('variant calling', CALLER),
    ('annotation', ANNOVAR_BUILD),
    ('reported absence rule', f'{REPORTED_RULE}: {RULE_TEXT[REPORTED_RULE]}'),
    ('reported set restrictions',
     f'SNV, AC_pass >= {MIN_AC_REPORTED}, call rate >= {MIN_CALL_RATE:.0%} of expected '
     f'ploidy, allele index resolved, autosomes and chrX'),
    ('file split rule', f'{FILE_SPLIT_RULE}: {RULE_TEXT[FILE_SPLIT_RULE]}'),
    ('expected allele number',
     f'autosome and PAR {AN_AUTO}, chrX outside PAR {AN_X_NONPAR}, '
     f'chrY outside PAR {AN_Y_NONPAR}'),
    ('chrY', 'excluded from allele-frequency and absence analysis'),
    ('enrichment', f'{ENRICHMENT_REF}, {ENRICHMENT_FOLD:.0f}x, AC >= {ENRICH_MIN_AC}, '
                   f'Wilson lower bound above the reference, '
                   f'{ENRICH_LO} < AF < {ENRICH_HI}'),
    ('reference-discordant rule',
     f'homozygous or hemizygous alternate >= {DISCORDANT_THRESHOLD:.0%}, '
     f'call rate >= {MIN_CALL_RATE:.0%}'),
    ('HWE flag', f'chi-square, 1 df, p < {HWE_ALPHA:g}, diploid regions only'),
    ('comparison panels', ', '.join(FREQ_PANELS)),
    ('correlation sampling ceiling', f'{CORR_MAX_N:,} sites per panel, seed 0'),
    ('MAF bins', 'lowest bin excluded from panel comparisons, reference '
                 'frequencies are rounded'),
    ('orthogonal confirmation', 'none performed'),
    ('dry run', 'yes, counts not final' if DRY_RUN_ROWS else 'no'),
    ('python', sys.version.split()[0]),
    ('numpy', np.__version__), ('pandas', pd.__version__),
    ('scipy', scipy.__version__), ('matplotlib', mpl.__version__),
], columns=['Key', 'Value'])

manifest_df = pd.DataFrame(MANIFEST).drop_duplicates(
    subset=['Stage', 'Role', 'File']).reset_index(drop=True)

pct = lambda v: f'{v / max(obs, 1) * 100:.2f}%'
print('\n' + '=' * 72)
print('VALUES FOR THE RESULTS TEXT')
print('=' * 72)
print(f'cohort                                  : {N_XX} XX, {N_XY} XY, n = {N_COHORT}')
print(f'variants in the annotated callset       : {counts["total"]:,}')
print(f'analysable (autosome or chrX, resolved) : {analysable:,}')
print(f'polymorphic in the cohort               : {obs:,}')
print(f'absent from the queried panels          : {n_reported:,} ({pct(n_reported)})')
print(f'  carrying a dbSNP/COSMIC/ClinVar record: {counts["reported_with_catalogue"]:,}')
print(f'  unrestricted upper bound (rule A)     : {state["rules"][REPORTED_RULE]:,} '
      f'({pct(state["rules"][REPORTED_RULE])})')
print(f'  absent indels, reported separately    : {counts["absent_indel"]:,}')
print(f'  absent SNV singletons, excluded       : {counts["absent_singleton"]:,}')
print(f'enriched vs {ENRICHMENT_REF:<20}: {counts["enriched_popmax"]:,} '
      f'(previous point rule gave {counts["enriched_pointrule"]:,})')
print(f'enriched vs gnomAD ALL                  : {counts["enriched_all"]:,}')
print(f'reference-discordant sites              : {counts["discordant"]:,}, '
      f'fully fixed {counts["fixed"]:,}')
if not disc_clin_df.empty:
    ben = disc_clin_df.loc[disc_clin_df['ClinVar class'] == 'Benign or likely benign',
                           'Percent of annotated'].sum()
    plp = disc_clin_df.loc[disc_clin_df['ClinVar class']
                           == 'Pathogenic or likely pathogenic', 'Sites'].sum()
    print(f'  with a ClinVar record                 : {int(disc_clin_df["Sites"].sum()):,}, '
          f'{ben:.2f}% benign or likely benign, {int(plp)} pathogenic')
print(f'rows excluded, allele index unresolved  : {par["unresolved"]:,}')
print(f'HWE p < {HWE_ALPHA:g}                          : {counts["hwe_extreme"]:,}')
if not corr_df.empty:
    b = corr_df.iloc[0]
    print(f'highest panel correlation               : {b["Panel"]} r = {b["Spearman r"]:.3f} '
          f'(descriptive only, see Methods)')
print('=' * 72)
print(f'\nnotebook 04 reads {os.path.basename(KNOWN_FILE)}')

In [ ]:
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side, Color
from openpyxl.utils import get_column_letter

HEAD_FILL = PatternFill(patternType='solid', fgColor=Color(rgb='FF1F3B4D'))
HEAD_FONT = Font(name='Arial', size=10, bold=True, color='FFFFFF')
BODY_FONT = Font(name='Arial', size=10)
THIN      = Side(style='thin', color='BFBFBF')

def style_sheet(ws, freeze='A2', center_from=1):
    """Header band, frozen panes, centred body, autosized columns."""
    ws.freeze_panes = freeze
    for cell in ws[1]:
        cell.fill = HEAD_FILL
        cell.font = HEAD_FONT
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = Border(bottom=THIN)
    ws.row_dimensions[1].height = 36
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.font = BODY_FONT
            cell.alignment = Alignment(
                horizontal='center' if cell.column >= center_from else 'left',
                vertical='center')
    for col in ws.iter_cols(max_col=ws.max_column):
        letter = get_column_letter(col[0].column)
        width = max((len(str(c.value)) for c in col if c.value is not None), default=8)
        ws.column_dimensions[letter].width = min(max(width + 3, 12), 44)


XLSX = os.path.join(OUT_DIR, f"Supplementary_Data_{SUPP_NUM['data']}_allele_frequencies.xlsx")

sheets = {
    '00_Summary'                              : summary_df,
    '01_allele_parsing'          : parsing_df,
    '02_absence_rules'           : rules_df,
    '03_RestrictionCascade'                   : cascade_df,
    '04_functional_class'        : func_table,
    '05_indel_sensitivity'       : indel_check,
    '06_Uncatalogued_TiTv'                    : titv_df,
    '07_Uncatalogued_spectrum'                : spec_df_cmp,
    '08_Uncatalogued_artefact_scan'           : artefact_df,
    '09_refdisc_functional'      : disc_func_df,
    '10_RefDiscordant_clinvar'                : disc_clin_df,
    '11_RefDiscordant_coding'                 : disc_coding_df,
    '12_RefDiscordant_MAF'                    : disc_maf_df,
    '13_panel_correlations'      : corr_df,
    '14_MAF_spectrum_cohort'     : spec_kz_df,
    '15_MAF_spectrum_panels'                  : spec_df,
    '16_AbsentKnown_functional'               : nk_full,
    '17_ChromDistribution'                    : chrom_df,
    '18_SexConcordance'                       : sex_concord.reset_index(names='IID'),
    '19_SexSummary'                           : sex_summary,
    '20_Preflight'                            : preflight,
    '21_SessionInfo'                          : session,
    '22_FileManifest'                         : manifest_df,
}

if os.path.exists(XLSX) and not OVERWRITE:
    XLSX = XLSX.replace('.xlsx', f'_{datetime.now():%Y%m%d_%H%M}.xlsx')

with pd.ExcelWriter(XLSX, engine='openpyxl') as xl:
    for name, frame in sheets.items():
        if frame is None or len(frame) == 0:
            print(f'  {name}: empty, skipped')
            continue
        out = frame
        if len(frame) > XL_ROW_LIMIT:
            spill = os.path.join(OUT_DIR, f'{name}.txt')
            frame.to_csv(spill, sep='\t', index=False)
            out = frame.head(XL_ROW_LIMIT)
            print(f'  {name}: {len(frame):,} rows, truncated; full table in '
                  f'{os.path.basename(spill)}')
        out.to_excel(xl, sheet_name=name[:31], index=False)

    for ws in xl.book.worksheets:
        left_only = ws.title.endswith(('_SessionInfo', '_FileManifest', '_Preflight'))
        style_sheet(ws, center_from=99 if left_only else 2)

        hdr = [c.value for c in ws[1]]
        for j, name in enumerate(hdr, start=1):
            if name is None: continue
            n = str(name).lower()
            if 'percent' in n or n.startswith('%'):
                fmt = '0.00'
            elif 'count' in n or 'sites' in n or 'variants' in n:
                fmt = '#,##0'
            else:
                continue
            for cell in ws.iter_cols(min_col=j, max_col=j, min_row=2):
                for c in cell:
                    c.number_format = fmt

print(f'\nsaved {os.path.basename(XLSX)}')

In [ ]:
from docx import Document
from docx.shared import Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT

SUPPL_TABLES = [
    ('tab_composition',
     'Composition of the annotated callset, the restriction cascade defining population-'
     'restricted variation, and classes defined independently of the cascade. ANNOVAR outputs '
     'one record per alternate allele, so a multiallelic position contributes more than one '
     'record. Indented rows are nested within the row above; the cascade is applied in the order '
     'shown, each step to the survivors of the previous one, and the number removed at each step '
     'is given in the definition. Percentages use the denominator named in the last column. The '
     'classes in the final block are defined independently of the cascade and of one another, so '
     'they overlap and do not sum. Reference-discordant and Hardy-Weinberg classes include indels; '
     'cascade rows below the single-nucleotide step do not.',
     summary_df),
    ('tab_parsing', 'Allele parsing and records excluded from allele counting.',
     parsing_df),
    ('tab_rules', 'Sensitivity of the count of population-restricted variation to its definition: '
                  'the four absence rules before and after the restriction cascade.',
     rules_df),
    ('tab_titv', 'Transition-to-transversion ratio of the two fractions of the reported set, '
                 'those carrying a catalogue record and those without one.',
     titv_df),
    ('tab_spectrum', 'Substitution spectrum of the two fractions of the reported set, six '
                     'strand-folded classes, with the difference in percentage points.',
     spec_df_cmp),
    ('tab_artefact', 'Cumulative filters applied to the uncatalogued fraction and the resulting '
                     'transition-to-transversion ratio after each.',
     artefact_df),
    ('tab_functional', 'Functional distribution of analysable records and of records absent from '
                       'the queried frequency panels, with log2 fold changes, delta-method '
                       'confidence intervals and Benjamini-Hochberg adjusted P values.',
     func_table),
    ('tab_indel', 'Absent-to-known ratio by exonic class: the sensitivity analysis motivating the '
                  'exclusion of insertions and deletions from the reported set.',
     indel_check),
    ('tab_refdisc', 'Reference-discordant sites by functional class.',
     disc_func_df),
    ('tab_correlation', 'Rank correlation between cohort allele frequencies and each reference '
                        'panel, computed on a fixed-seed reservoir sample of the shared sites.',
     corr_df),
    ('tab_maf', 'Minor-allele-frequency spectrum of the cohort.',
     spec_kz_df),
]
SUPPL_TABLES = sorted(SUPPL_TABLES, key=lambda t: SUPP_NUM[t[0]])

def _fmt(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ''
    if isinstance(v, (int, np.integer)):
        return f'{int(v):,}'
    if isinstance(v, (bool, np.bool_)):
        return 'yes' if v else 'no'
    if isinstance(v, (float, np.floating)):
        if float(v).is_integer() and abs(v) >= 1:
            return f'{int(v):,}'
        return f'{v:,.3f}'.rstrip('0').rstrip('.')
    return str(v)

def add_table(doc, df, caption, tag=None):
    p = doc.add_paragraph()
    r = p.add_run(caption if tag is None else f'{tag}  {caption}')
    r.bold = True
    r.font.size = Pt(9)
    r.font.name = 'Arial'
    p.paragraph_format.space_after = Pt(4)

    t = doc.add_table(rows=1, cols=len(df.columns))
    t.style = 'Table Grid'
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    t.autofit = True
    for j, col in enumerate(df.columns):
        c = t.rows[0].cells[j]
        c.text = ''
        run = c.paragraphs[0].add_run(str(col))
        run.bold = True
        run.font.size = Pt(8)
        run.font.name = 'Arial'
        c.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
    for _, row in df.iterrows():
        cells = t.add_row().cells
        for j, v in enumerate(row):
            cells[j].text = ''
            run = cells[j].paragraphs[0].add_run(_fmt(v))
            run.font.size = Pt(8)
            run.font.name = 'Arial'
            cells[j].paragraphs[0].alignment = (
                WD_ALIGN_PARAGRAPH.LEFT if j == 0 else WD_ALIGN_PARAGRAPH.CENTER)
    doc.add_paragraph().paragraph_format.space_after = Pt(10)

doc = Document()
for s in doc.sections:
    s.left_margin = s.right_margin = Inches(0.7)
h = doc.add_paragraph()
hr = h.add_run('Supplementary Tables 03')
hr.bold = True
hr.font.size = Pt(12)
hr.font.name = 'Arial'

written = 0
for key, cap, df in SUPPL_TABLES:
    num = SUPP_NUM[key]
    if df is None or len(df) == 0:
        print(f'  Supplementary Table {num}: empty, skipped')
        continue
    add_table(doc, df, cap, tag=f'Supplementary Table {num}.')
    written += 1
    print(f'  Supplementary Table {num} ({key}): {df.shape[0]} rows x {df.shape[1]} cols')

DOCX = os.path.join(OUT_DIR, 'Supplementary_Tables_03.docx')
doc.save(DOCX)
print(f'saved {os.path.basename(DOCX)}: {written} tables')